# AetherTTS — Phase 1 流式主干对比实验（Google Colab 一键运行版）

本 Notebook 用于在 Google Colab GPU 上**端到端跑通 Phase 1 控制变量实验**，完成：
1. **数据下载与 Google 云盘缓存**：自动将单人语料缓存到 Google Drive（二次运行跳过下载），并暂存到本地高速 SSD 进行训练。
2. **多模型同预算训练与同步云盘**：在相同参数规模（~4.5–5.5M 声学主干）和相同有限右前瞻（默认 `L = 4` 音素）下，依次训练并对比 5 个模型：
   - `E0_offline_bitransformer`：非流式全双向 Transformer（音质上界参考）
   - `E1_conv_unigru_L4`：集中右前瞻 + Causal ConvNeXt + 单向 GRU（极简基线）
   - `E2_swa_L4`：集中右前瞻 + 滑动窗口因果注意力 SWA（RoPE + 有界 KV Cache）+ Causal ConvNeXt
   - `E3_ssm_mamba_L4`：集中右前瞻 + 纯 PyTorch Selective SSM（Mamba 风格，O(1) 状态，零 CUDA 编译依赖）
   - `E4_rwkv_L4`：集中右前瞻 + 纯 PyTorch 数据依赖线性注意力 WKV（RWKV 风格，O(1) 状态）
3. **流式↔离线数学等价硬门禁**：训练前自动断言 `E1`–`E4` 与 `CausalVocosLite`（~3.5M 24 kHz 因果声码器）逐块流式输出与整句离线前向误差 `< 1e-4`。
4. **批量生成、自动备份到 Google 云盘、多维客观打分 & 盲听**：
   - 将所有模型权重（FP32 + INT8）、离线与真流式生成的 24 kHz WAV、评测报表与对比图全部自动拷贝到 `/content/drive/MyDrive/AetherTTS_Exp/`。
   - 自动评测：**CER/WER/MER（可懂度）+ UTMOS（自然度）+ SQUIM PESQ/SI-SDR/STOI（信号质量）+ 2–8kHz 高频嗡鸣诊断 + 声纹相似度 SIM + 流式接缝爆音检测 + 纯 CPU 2 线程 RTF/TTFA**，并输出总排行榜。

> **建议运行时**：Colab GPU（L4 / A100 最佳，T4 亦可）。首次验证建议保持 `EXPERIMENT_PROFILE = "quick_smoke"`（约 15–20 分钟走完全流程），确认无误后再切到 `"standard_phase1"`（约 2–3 小时）或 `"full_quality"`。


## 1. 挂载 Google Drive 与全局实验配置


In [ ]:
import os
import sys
import json
import math
import time
import shutil
import random
import tarfile
import warnings
from pathlib import Path
from dataclasses import dataclass, asdict

import torch

# 1) 挂载 Google Drive（若在非 Colab 环境测试则自动回退到本地目录）
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/AetherTTS_Exp")
    LOCAL_WORK = Path("/content/aethertts_work")
else:
    DRIVE_ROOT = Path("./aethertts_drive_mock")
    LOCAL_WORK = Path("./aethertts_work")

for p in [
    DRIVE_ROOT / "datasets" / "raw",
    DRIVE_ROOT / "vocoder",
    DRIVE_ROOT / "runs",
    DRIVE_ROOT / "eval_samples",
    DRIVE_ROOT / "reports",
    LOCAL_WORK / "data",
    LOCAL_WORK / "cache",
    LOCAL_WORK / "runs",
]:
    p.mkdir(parents=True, exist_ok=True)

# ==============================================================================
# 用户核心配置区（只需改这里！）
# ==============================================================================
# 实验档位：
#   "quick_smoke"     : ~15-20 min (2h数据，快速验证全流程、流式等价、云盘同步与打分)
#   "standard_phase1" : ~2-3h (8h数据，足够听出 5 个主干的明显优劣差异)
#   "full_quality"    : ~6-8h (12h-22h数据，单音色高保真收敛)
EXPERIMENT_PROFILE = "standard_phase1"

# 数据集选择：
#   "bznsyp"           : 标贝中文单人女声 12h (48kHz, HF直下, 调研阶段测中文多音字/变调首选; 非商用)
#   "ljspeech"         : LJSpeech 英文单人女声 24h (22.05kHz, Public Domain 商用安全)
#   "custom_bilingual" : 读取 DRIVE_ROOT/datasets/custom_bilingual/ (metadata.csv + wavs/, 中英混读单音色)
DATASET_SOURCE = "bznsyp"

# 声码器模式：
#   "both"             : 训练自研 ~3.5M CausalVocosLite (测真流式/CPU RTF) + 加载预训练 Vocos 24kHz 作上界对照
#   "pretrained_vocos" : 跳过声码器训练，直接用预训练 Vocos 24kHz 解码（最快对比声学主干）
#   "causal_vocos_lite": 仅训练并使用自研 ~3.5M CausalVocosLite
VOCODER_MODE = "both"

# 本次要依次训练并打分的模型列表：(实验名, 主干类型, 文本右前瞻音素数 L)
# 可按需增删，例如追加 ("E2_swa_L0", "swa", 0) 或 ("E2_swa_L8", "swa", 8) 做右前瞻消融
EXPERIMENTS = [
    ("E0_offline_bitransformer", "offline_transformer", 0),
    ("E1_conv_unigru_L4",        "conv_gru",            4),
    ("E2_swa_L4",                "swa",                 4),
    ("E3_ssm_mamba_L4",          "ssm",                 4),
    ("E4_rwkv_L4",               "rwkv",                4),
]

# 断点续训开关：如果 Google Drive 里已有训练完成的 best.pt，是否直接跳过训练进入评测？
SKIP_IF_DONE_ON_DRIVE = True

# 档位详细超参（严格控量控时，防止训练失控）
PROFILE_PRESETS = {
    "quick_smoke": {
        "max_train_hours": 2.0,
        "val_utts": 40,
        "vocoder_stage1_steps": 1500,
        "vocoder_stage2_steps": 800,
        "acoustic_steps": 1500,
        "batch_size": 16,
        "eval_max_utts": 12,
    },
    "standard_phase1": {
        "max_train_hours": 8.0,
        "val_utts": 80,
        "vocoder_stage1_steps": 12000,
        "vocoder_stage2_steps": 6000,
        "acoustic_steps": 15000,
        "batch_size": 24,
        "eval_max_utts": 25,
    },
    "full_quality": {
        "max_train_hours": 20.0,
        "val_utts": 100,
        "vocoder_stage1_steps": 35000,
        "vocoder_stage2_steps": 15000,
        "acoustic_steps": 40000,
        "batch_size": 32,
        "eval_max_utts": 50,
    },
}

CFG = {
    "seed": 42,
    "sample_rate": 24000,
    "n_fft": 1024,
    "hop_length": 256,
    "win_length": 1024,
    "n_mels": 100,
    "min_audio_sec": 0.8,
    "max_audio_sec": 12.0,
    "d_model": 256,
    "k_left": 3,
    "swa_window": 48,
    "stream_text_chunk": 3,   # 流式评测每次送入的音素块大小
    "stream_mel_chunk": 16,   # 声码器流式每次吐出帧数 (16 * 256 / 24000 ≈ 170 ms)
    "lr_acoustic": 4e-4,
    "lr_vocoder": 2e-4,
    "profile": EXPERIMENT_PROFILE,
    "dataset_source": DATASET_SOURCE,
    "vocoder_mode": VOCODER_MODE,
    **PROFILE_PRESETS[EXPERIMENT_PROFILE],
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_DTYPE = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16

print(f"Device       : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
print(f"Drive Root   : {DRIVE_ROOT}")
print(f"Local Work   : {LOCAL_WORK}")
print(f"Profile      : {EXPERIMENT_PROFILE} -> max_train_hours={CFG['max_train_hours']}h, acoustic_steps={CFG['acoustic_steps']}")
print(f"Dataset      : {DATASET_SOURCE}")
print(f"Vocoder Mode : {VOCODER_MODE}")
print(f"Models       : {[x[0] for x in EXPERIMENTS]}")


## 2. 安装依赖（Colab 环境）


In [ ]:
!pip -q install --upgrade pypinyin g2p-en soundfile librosa scipy einops tqdm pandas matplotlib huggingface_hub jiwer vocos faster-whisper resemblyzer
# 可选：中文评测增强（SenseVoice / Paraformer），安装失败不影响主流程
!pip -q install funasr modelscope || true


## 3. 基础设施：音频特征提取（24 kHz 100-Mel 与 Vocos 对齐）与统一中英混读前端


In [ ]:
import re
import unicodedata
import numpy as np
import soundfile as sf
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from pypinyin import pinyin, Style

try:
    from g2p_en import G2p
    import nltk
    for res in ["averaged_perceptron_tagger_eng", "averaged_perceptron_tagger", "cmudict"]:
        try:
            nltk.download(res, quiet=True)
        except Exception:
            pass
    _EN_G2P = G2p()
except Exception as e:
    print(f"[Warn] g2p_en unavailable ({e}), falling back to character-level EN G2P.")
    _EN_G2P = None


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(CFG["seed"])


def sync_file_to_drive(local_path: Path, drive_path: Path):
    drive_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = drive_path.with_suffix(drive_path.suffix + ".tmp")
    shutil.copy2(local_path, tmp_path)
    os.replace(tmp_path, drive_path)


def sync_dir_to_drive(local_dir: Path, drive_dir: Path):
    drive_dir.mkdir(parents=True, exist_ok=True)
    for src in local_dir.rglob("*"):
        if src.is_file():
            rel = src.relative_to(local_dir)
            sync_file_to_drive(src, drive_dir / rel)


# ------------------------------------------------------------------------------
# 24 kHz 100-bin Log-Mel 提取器（与 charactr/vocos-mel-24khz 100% 参数对齐）
# ------------------------------------------------------------------------------
class VocosCompatibleMel(nn.Module):
    def __init__(self, sample_rate=24000, n_fft=1024, hop_length=256, n_mels=100):
        super().__init__()
        self.mel_spec = torchaudio.transforms.MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=n_fft,
            hop_length=hop_length,
            n_mels=n_mels,
            center=True,
            power=1,
        )

    def forward(self, wav: torch.Tensor) -> torch.Tensor:
        # wav: [B, T] or [T] -> returns [B, n_mels, F] or [n_mels, F]
        squeeze = False
        if wav.dim() == 1:
            wav = wav.unsqueeze(0)
            squeeze = True
        mel = self.mel_spec(wav)
        log_mel = torch.log(torch.clamp(mel, min=1e-7))
        return log_mel.squeeze(0) if squeeze else log_mel


MEL_EXTRACTOR = VocosCompatibleMel(
    sample_rate=CFG["sample_rate"],
    n_fft=CFG["n_fft"],
    hop_length=CFG["hop_length"],
    n_mels=CFG["n_mels"],
)


def load_and_normalize_wav(path: Path, target_sr: int = 24000) -> np.ndarray:
    wav, sr = sf.read(str(path), dtype="float32")
    if wav.ndim > 1:
        wav = wav.mean(axis=-1)
    if sr != target_sr:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=target_sr)
    # 裁剪首尾过长静音，保留 ~60 ms 自然起止缓冲
    wav_trimmed, idx = librosa.effects.trim(wav, top_db=35, frame_length=1024, hop_length=256)
    pad = int(0.06 * target_sr)
    start = max(0, idx[0] - pad)
    end = min(len(wav), idx[1] + pad)
    wav = wav[start:end]
    # 响度/峰值归一化（防削波）
    rms = np.sqrt(np.mean(wav ** 2) + 1e-8)
    wav = wav * (0.10 / max(rms, 1e-4))
    peak = np.max(np.abs(wav)) + 1e-8
    if peak > 0.92:
        wav = wav * (0.92 / peak)
    return wav.astype(np.float32)


def extract_f0_and_energy(wav: np.ndarray, sr: int = 24000, hop_length: int = 256, n_frames: int = None):
    # 快速稳定的基频提取 (librosa.yin) + 线性插值 log-F0
    f0 = librosa.yin(
        wav, fmin=65.0, fmax=600.0, sr=sr, frame_length=1024, hop_length=hop_length, center=True
    ).astype(np.float32)
    # 能量包络（帧 RMS）
    rms = librosa.feature.rms(y=wav, frame_length=1024, hop_length=hop_length, center=True)[0].astype(np.float32)
    if n_frames is not None:
        if len(f0) < n_frames:
            f0 = np.pad(f0, (0, n_frames - len(f0)), mode="edge")
            rms = np.pad(rms, (0, n_frames - len(rms)), mode="edge")
        else:
            f0 = f0[:n_frames]
            rms = rms[:n_frames]
    # 静音帧清零基频并对有声段做连续 log-F0 插值
    uv = (rms > 0.008) & (f0 > 70.0) & (f0 < 580.0)
    log_f0 = np.zeros_like(f0, dtype=np.float32)
    if uv.any():
        voiced_idx = np.where(uv)[0]
        voiced_vals = np.log(f0[voiced_idx])
        log_f0 = np.interp(np.arange(len(f0)), voiced_idx, voiced_vals).astype(np.float32)
    log_energy = np.log(np.clip(rms, 1e-5, None)).astype(np.float32)
    return log_f0, log_energy, uv.astype(np.float32)


# ------------------------------------------------------------------------------
# 统一中英混读音素前端 (ZH 声母/带调韵母 + EN ARPAbet + 韵律/标点 + LangID)
# ------------------------------------------------------------------------------
ZH_INITIALS = [
    "b", "p", "m", "f", "d", "t", "n", "l", "g", "k", "h",
    "j", "q", "x", "zh", "ch", "sh", "r", "z", "c", "s", "y", "w",
]
ZH_FINALS_BASE = [
    "a", "o", "e", "i", "u", "v", "ai", "ei", "ui", "ao", "ou", "iu", "ie", "ve", "er",
    "an", "en", "in", "un", "vn", "ang", "eng", "ing", "ong",
    "ia", "iao", "ian", "iang", "iong", "ua", "uo", "uai", "uan", "uang", "ue", "van",
]
EN_ARPABET = [
    "AA0", "AA1", "AA2", "AE0", "AE1", "AE2", "AH0", "AH1", "AH2", "AO0", "AO1", "AO2",
    "AW0", "AW1", "AW2", "AY0", "AY1", "AY2", "B", "CH", "D", "DH",
    "EH0", "EH1", "EH2", "ER0", "ER1", "ER2", "EY0", "EY1", "EY2", "F", "G", "HH",
    "IH0", "IH1", "IH2", "IY0", "IY1", "IY2", "JH", "K", "L", "M", "N", "NG",
    "OW0", "OW1", "OW2", "OY0", "OY1", "OY2", "P", "R", "S", "SH", "T", "TH",
    "UH0", "UH1", "UH2", "UW0", "UW1", "UW2", "V", "W", "Y", "Z", "ZH",
]
SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<unk>", "<sp>", ",", ".", "?", "!", ";", ":", "#1", "#2", "#3"]


class BilingualPhonemeFrontend:
    """
    统一中英混读前端（含轻量数字/百分比正则化、中文声韵调拆解与三声变调、英文 ARPAbet G2P）。
    输出 (phoneme_ids, lang_ids)，其中 lang_id: 0=SPECIAL/PUNC, 1=ZH, 2=EN。
    """
    def __init__(self):
        vocab = list(SPECIAL_TOKENS)
        vocab += [f"zh_{ini}" for ini in ZH_INITIALS]
        for fin in ZH_FINALS_BASE:
            for tone in ["1", "2", "3", "4", "5"]:
                vocab.append(f"zh_{fin}{tone}")
        vocab += [f"en_{ph}" for ph in EN_ARPABET]
        self.vocab = vocab
        self.tok2id = {t: i for i, t in enumerate(vocab)}
        self.id2tok = {i: t for i, t in enumerate(vocab)}
        self.pad_id = self.tok2id["<pad>"]
        self.bos_id = self.tok2id["<bos>"]
        self.eos_id = self.tok2id["<eos>"]
        self.unk_id = self.tok2id["<unk>"]
        self.sp_id = self.tok2id["<sp>"]

    @property
    def vocab_size(self) -> int:
        return len(self.vocab)

    @staticmethod
    def _num_to_zh(num_str: str) -> str:
        digits = "零一二三四五六七八九"
        if "." in num_str:
            int_p, dec_p = num_str.split(".", 1)
            return BilingualPhonemeFrontend._num_to_zh(int_p) + "点" + "".join(digits[int(c)] for c in dec_p if c.isdigit())
        n = int(num_str)
        if len(num_str) == 4 and 1900 <= n <= 2099:
            return "".join(digits[int(c)] for c in num_str)
        if n == 0:
            return "零"
        if n < 10:
            return digits[n]
        if n < 20:
            return "十" + ("" if n == 10 else digits[n % 10])
        if n < 100:
            return digits[n // 10] + "十" + ("" if n % 10 == 0 else digits[n % 10])
        return "".join(digits[int(c)] for c in num_str)

    def normalize_text(self, text: str) -> str:
        text = unicodedata.normalize("NFKC", text).strip()
        # 标点统一
        punc_map = {"，": ",", "。": ".", "？": "?", "！": "!", "；": ";", "：": ":", "、": ",", "—": ",", "…": "."}
        for k, v in punc_map.items():
            text = text.replace(k, v)
        # 若含中文，做轻量中文数字/百分比正则化（调研期轻量替代 WeTextProcessing）
        has_zh = bool(re.search(r"[\u4e00-\u9fff]", text))
        if has_zh:
            text = re.sub(r"(\d+(?:\.\d+)?)%", lambda m: "百分之" + self._num_to_zh(m.group(1)), text)
            text = re.sub(r"\d+(?:\.\d+)?", lambda m: self._num_to_zh(m.group(0)), text)
        return text

    @staticmethod
    def _split_zh_syllable(syl: str):
        # syl 形如 "zhong1", "nv3", "er2", "a5"
        syl = syl.lower().replace("ü", "v").replace("u:", "v")
        tone = syl[-1] if syl and syl[-1] in "12345" else "5"
        body = syl[:-1] if syl and syl[-1] in "12345" else syl
        if not body:
            return []
        ini = ""
        for cand in sorted(ZH_INITIALS, key=len, reverse=True):
            if body.startswith(cand) and len(body) > len(cand):
                ini = cand
                body = body[len(cand):]
                break
        if body == "u" and ini in ("j", "q", "x", "y"):
            body = "v"
        elif body == "ue" and ini in ("j", "q", "x", "y"):
            body = "ve"
        elif body == "uan" and ini in ("j", "q", "x", "y"):
            body = "van"
        elif body == "un" and ini in ("j", "q", "x", "y"):
            body = "vn"
        toks = []
        if ini:
            toks.append(f"zh_{ini}")
        toks.append(f"zh_{body}{tone}")
        return toks

    def _zh_span_to_phonemes(self, span: str):
        # pypinyin 转带调拼音 + 简单三声变调（连续两个三声，前者变二声）
        pys = [p[0] for p in pinyin(span, style=Style.TONE3, neutral_tone_with_five=True, errors="ignore") if p]
        for i in range(len(pys) - 1):
            if pys[i].endswith("3") and pys[i + 1].endswith("3"):
                pys[i] = pys[i][:-1] + "2"
        out = []
        for s in pys:
            out.extend(self._split_zh_syllable(s))
        return out

    def _en_span_to_phonemes(self, span: str):
        if _EN_G2P is not None:
            raw = _EN_G2P(span)
            out = []
            for p in raw:
                if p == " ":
                    out.append("<sp>")
                elif f"en_{p}" in self.tok2id:
                    out.append(f"en_{p}")
            return out
        # 兜底映射
        return ["<sp>"]

    def text_to_ids(self, text: str, prosody_pinyin: list = None):
        """返回 (phoneme_ids: List[int], lang_ids: List[int])"""
        tokens = ["<bos>"]
        langs = [0]

        if prosody_pinyin is not None:
            # 标贝自带精确拼音+韵律边界路径
            for item in prosody_pinyin:
                if item in self.tok2id:
                    tokens.append(item)
                    langs.append(0)
                else:
                    for ph in self._split_zh_syllable(item):
                        tokens.append(ph if ph in self.tok2id else "<unk>")
                        langs.append(1)
        else:
            norm = self.normalize_text(text)
            # 按中文段 / 英文词段 / 标点韵律段切分（天然支持中英混读）
            pattern = re.compile(r"([\u4e00-\u9fff]+|[A-Za-z][A-Za-z'\-]*|#1|#2|#3|[,\.\?!;:])")
            prev_lang = 0
            for m in pattern.finditer(norm):
                seg = m.group(0)
                if re.match(r"^[\u4e00-\u9fff]+$", seg):
                    if prev_lang == 2:
                        tokens.append("<sp>"); langs.append(0)
                    for ph in self._zh_span_to_phonemes(seg):
                        tokens.append(ph if ph in self.tok2id else "<unk>")
                        langs.append(1)
                    prev_lang = 1
                elif re.match(r"^[A-Za-z]", seg):
                    if prev_lang in (1, 2):
                        tokens.append("<sp>"); langs.append(0)
                    for ph in self._en_span_to_phonemes(seg):
                        tokens.append(ph if ph in self.tok2id else "<unk>")
                        langs.append(2 if ph.startswith("en_") else 0)
                    prev_lang = 2
                else:
                    if seg in self.tok2id:
                        tokens.append(seg)
                        langs.append(0)
                    prev_lang = 0

        tokens.append("<eos>")
        langs.append(0)
        ids = [self.tok2id.get(t, self.unk_id) for t in tokens]
        return ids, langs


FRONTEND = BilingualPhonemeFrontend()
print(f"Unified Bilingual Vocab Size: {FRONTEND.vocab_size}")
for demo in ["你好，请问今天北京的天气怎么样？", "Hello world, how is your API latency today?", "帮我把 Wi-Fi 切换到 5G，谢谢！"]:
    pids, lids = FRONTEND.text_to_ids(demo)
    print(f"  {demo} -> {len(pids)} phonemes")


## 4. 下载数据到 Google 云盘缓存并预处理（严格控量）


In [ ]:
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download


def download_and_prepare_dataset(cfg):
    src = cfg["dataset_source"]
    drive_raw_dir = DRIVE_ROOT / "datasets" / "raw"
    local_raw_dir = LOCAL_WORK / "data" / src
    local_feat_dir = LOCAL_WORK / "cache" / f"{src}_max{cfg['max_train_hours']}h"
    drive_manifest_cache = DRIVE_ROOT / "datasets" / f"{src}_manifest_max{cfg['max_train_hours']}h.json"

    local_raw_dir.mkdir(parents=True, exist_ok=True)
    local_feat_dir.mkdir(parents=True, exist_ok=True)

    items = []  # list of dict(utt_id, wav_path, text, prosody_pinyin)

    if src == "bznsyp":
        drive_archive = drive_raw_dir / "BZNSYP.tar.bz2"
        if not drive_archive.exists():
            print("[Download] 正在从 HuggingFace 下载标贝 BZNSYP (12h) 到 Google Drive 缓存...")
            dl_path = hf_hub_download(
                repo_id="openspeech/BZNSYP",
                filename="BZNSYP.tar.bz2",
                repo_type="model",
                local_dir=str(LOCAL_WORK / "tmp_dl"),
            )
            sync_file_to_drive(Path(dl_path), drive_archive)
            print(f"[Drive Cache] 已缓存至 {drive_archive}")
        else:
            print(f"[Drive Cache Hit] 直接复用云盘压缩包: {drive_archive}")

        marker = local_raw_dir / ".extracted"
        if not marker.exists():
            print("[Extract] 解压 BZNSYP 到本地高速 SSD...")
            with tarfile.open(drive_archive, "r:bz2") as tf:
                tf.extractall(local_raw_dir)
            marker.touch()

        # 查找标注文件与 Wave 目录
        txt_candidates = list(local_raw_dir.rglob("*000001-010000.txt"))
        assert txt_candidates, f"未找到 000001-010000.txt，请检查解压目录 {local_raw_dir}"
        label_file = txt_candidates[0]
        wav_dirs = [p for p in local_raw_dir.rglob("Wave") if p.is_dir()]
        wav_dir = wav_dirs[0]

        raw_bytes = label_file.read_bytes()
        try:
            lines = raw_bytes.decode("utf-8").splitlines()
        except UnicodeDecodeError:
            lines = raw_bytes.decode("gb18030", errors="ignore").splitlines()

        for i in range(0, len(lines) - 1, 2):
            line_txt = lines[i].strip()
            line_py = lines[i + 1].strip()
            if not line_txt or "\t" not in line_txt:
                continue
            utt_id, text_with_prosody = line_txt.split("\t", 1)
            wav_path = wav_dir / f"{utt_id.strip()}.wav"
            if not wav_path.exists():
                continue
            clean_text = re.sub(r"#[1-4]", "", text_with_prosody).strip()
            # 保留原始拼音列表，同时利用文本中标点补充停顿
            py_list = line_py.split()
            items.append({
                "utt_id": utt_id.strip(),
                "wav_path": str(wav_path),
                "text": clean_text,
                "prosody_pinyin": py_list,
            })

    elif src == "ljspeech":
        drive_archive = drive_raw_dir / "LJSpeech-1.1.tar.bz2"
        if not drive_archive.exists():
            print("[Download] 正在下载 LJSpeech-1.1 (Public Domain) 到 Google Drive 缓存...")
            tmp_tar = LOCAL_WORK / "LJSpeech-1.1.tar.bz2"
            os.system(f'wget -q --show-progress -O "{tmp_tar}" "https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2"')
            sync_file_to_drive(tmp_tar, drive_archive)
            tmp_tar.unlink(missing_ok=True)
            print(f"[Drive Cache] 已缓存至 {drive_archive}")
        else:
            print(f"[Drive Cache Hit] 直接复用云盘压缩包: {drive_archive}")

        marker = local_raw_dir / ".extracted"
        if not marker.exists():
            print("[Extract] 解压 LJSpeech-1.1 到本地高速 SSD...")
            with tarfile.open(drive_archive, "r:bz2") as tf:
                tf.extractall(local_raw_dir)
            marker.touch()

        lj_root = local_raw_dir / "LJSpeech-1.1"
        meta_path = lj_root / "metadata.csv"
        for line in meta_path.read_text(encoding="utf-8").splitlines():
            parts = line.strip().split("|")
            if len(parts) < 2:
                continue
            utt_id = parts[0]
            text = parts[2] if len(parts) >= 3 else parts[1]
            wav_path = lj_root / "wavs" / f"{utt_id}.wav"
            if wav_path.exists():
                items.append({
                    "utt_id": utt_id,
                    "wav_path": str(wav_path),
                    "text": text,
                    "prosody_pinyin": None,
                })

    elif src == "custom_bilingual":
        custom_dir = DRIVE_ROOT / "datasets" / "custom_bilingual"
        meta_path = custom_dir / "metadata.csv"
        assert meta_path.exists(), f"请先将单音色双语数据放到 {custom_dir} (包含 metadata.csv 与 wavs/)"
        for line in meta_path.read_text(encoding="utf-8").splitlines():
            parts = line.strip().split("|")
            if len(parts) < 2:
                continue
            utt_id, text = parts[0].strip(), parts[1].strip()
            wav_path = custom_dir / "wavs" / f"{utt_id}.wav"
            if wav_path.exists():
                items.append({
                    "utt_id": utt_id,
                    "wav_path": str(wav_path),
                    "text": text,
                    "prosody_pinyin": None,
                })
    else:
        raise ValueError(f"Unsupported DATASET_SOURCE: {src}")

    print(f"[Dataset] 扫描到原始条目: {len(items)} 条，开始按 max_train_hours={cfg['max_train_hours']}h 严控预处理...")

    # 固定随机打乱，划分 val 与受控 train
    rng = random.Random(cfg["seed"])
    rng.shuffle(items)

    processed = []
    total_sec = 0.0
    max_total_sec = (cfg["max_train_hours"] + 0.5) * 3600.0  # 含 val 预留

    mel_sum = np.zeros(cfg["n_mels"], dtype=np.float64)
    mel_sq = np.zeros(cfg["n_mels"], dtype=np.float64)
    mel_cnt = 0
    f0_vals = []
    e_vals = []

    for it in tqdm(items, desc="Extract 24kHz Mel+F0+Energy"):
        if total_sec >= max_total_sec:
            break
        feat_file = local_feat_dir / f"{it['utt_id']}.pt"
        if feat_file.exists():
            d = torch.load(feat_file, map_location="cpu", weights_only=False)
            dur_sec = float(d["wav"].shape[0]) / cfg["sample_rate"]
            mel_np = d["mel"].numpy()
            f0_np = d["log_f0"].numpy()
            e_np = d["log_energy"].numpy()
            uv_np = d["uv"].numpy()
        else:
            try:
                wav = load_and_normalize_wav(Path(it["wav_path"]), cfg["sample_rate"])
            except Exception:
                continue
            dur_sec = len(wav) / cfg["sample_rate"]
            if dur_sec < cfg["min_audio_sec"] or dur_sec > cfg["max_audio_sec"]:
                continue
            pids, lids = FRONTEND.text_to_ids(it["text"], prosody_pinyin=it["prosody_pinyin"])
            if len(pids) < 4 or len(pids) > 220:
                continue
            wav_t = torch.from_numpy(wav)
            mel_t = MEL_EXTRACTOR(wav_t)  # [100, F]
            n_frames = mel_t.shape[-1]
            if n_frames < len(pids) * 2:
                continue
            f0_np, e_np, uv_np = extract_f0_and_energy(wav, cfg["sample_rate"], cfg["hop_length"], n_frames)
            d = {
                "utt_id": it["utt_id"],
                "text": it["text"],
                "phoneme_ids": torch.tensor(pids, dtype=torch.long),
                "lang_ids": torch.tensor(lids, dtype=torch.long),
                "mel": mel_t.to(torch.float32),
                "log_f0": torch.from_numpy(f0_np),
                "log_energy": torch.from_numpy(e_np),
                "uv": torch.from_numpy(uv_np),
                "wav": wav_t,
            }
            torch.save(d, feat_file)
            mel_np = mel_t.numpy()

        total_sec += dur_sec
        mel_sum += mel_np.sum(axis=-1)
        mel_sq += (mel_np ** 2).sum(axis=-1)
        mel_cnt += mel_np.shape[-1]
        if uv_np.sum() > 0:
            f0_vals.append(f0_np[uv_np > 0.5])
        e_vals.append(e_np)

        processed.append({
            "utt_id": it["utt_id"],
            "text": it["text"],
            "feat_path": str(feat_file),
            "dur_sec": dur_sec,
            "n_phonemes": int(len(d["phoneme_ids"])),
            "n_frames": int(d["mel"].shape[-1]),
        })

    mel_mean = (mel_sum / max(mel_cnt, 1)).astype(np.float32)
    mel_std = np.sqrt(np.maximum(mel_sq / max(mel_cnt, 1) - mel_mean ** 2, 1e-4)).astype(np.float32)
    all_f0 = np.concatenate(f0_vals) if f0_vals else np.array([5.0], dtype=np.float32)
    all_e = np.concatenate(e_vals) if e_vals else np.array([-3.0], dtype=np.float32)

    stats = {
        "mel_mean": mel_mean.tolist(),
        "mel_std": mel_std.tolist(),
        "f0_mean": float(np.mean(all_f0)),
        "f0_std": float(max(np.std(all_f0), 1e-3)),
        "e_mean": float(np.mean(all_e)),
        "e_std": float(max(np.std(all_e), 1e-3)),
    }

    n_val = min(cfg["val_utts"], max(10, len(processed) // 10))
    val_list = processed[:n_val]
    train_list = processed[n_val:]

    meta = {
        "source": src,
        "total_hours": round(total_sec / 3600.0, 3),
        "train_count": len(train_list),
        "val_count": len(val_list),
        "stats": stats,
        "train": train_list,
        "val": val_list,
    }
    local_manifest = local_feat_dir / "manifest.json"
    local_manifest.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    sync_file_to_drive(local_manifest, drive_manifest_cache)

    print(f"[Ready] 有效数据 {len(processed)} 句（共 {total_sec/3600:.2f} 小时）| Train={len(train_list)} | Val={len(val_list)}")
    return meta


DATASET_META = download_and_prepare_dataset(CFG)


## 5. 核心模型实现：5 大流式/非流式声学主干（`E0`–`E4`）+ 单调对齐器 + 因果声码器 `CausalVocosLite`

设计要点：
- **集中式有限右前瞻层（`CentralizedLookaheadBlock`）**：仅第 0 层允许看右边 `L` 个音素，后续所有层严格因果（0 额外右前瞻），彻底杜绝多层前瞻膨胀。
- **全模型双路径接口**：`E1`–`E4` 及 `CausalVocosLite` 同时提供并行训练 `forward()` 和带显式缓存状态的真流式 `forward_step()`，下一节立即做数值等价门禁测试。


In [ ]:
# ==============================================================================
# 5.1 集中式有限右前瞻层 & 严格因果 ConvNeXt 块（均支持 100% 数值等价流式缓存）
# ==============================================================================
class CentralizedLookaheadBlock(nn.Module):
    """
    全网唯一具备右前瞻的层：每个位置 t 严格只看 [t - k_left, t + lookahead]。
    """
    def __init__(self, d_model: int, k_left: int = 3, lookahead: int = 4, mult: int = 2):
        super().__init__()
        self.d_model = d_model
        self.k_left = k_left
        self.lookahead = lookahead
        self.kernel_size = k_left + 1 + lookahead
        self.dwconv = nn.Conv1d(d_model, d_model, kernel_size=self.kernel_size, groups=d_model)
        self.norm = nn.LayerNorm(d_model)
        self.pw1 = nn.Linear(d_model, d_model * mult)
        self.pw2 = nn.Linear(d_model * mult, d_model)

    def _apply_body(self, conv_out: torch.Tensor, residual: torch.Tensor) -> torch.Tensor:
        h = self.norm(conv_out.transpose(1, 2))
        h = self.pw2(F.gelu(self.pw1(h)))
        return residual + h

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, T, D]
        x_pad = F.pad(x.transpose(1, 2), (self.k_left, self.lookahead))
        conv_out = self.dwconv(x_pad)
        return self._apply_body(conv_out, x)

    def init_state(self, batch_size: int, device, dtype):
        return torch.zeros(batch_size, self.k_left, self.d_model, device=device, dtype=dtype)

    def forward_step(self, x_new: torch.Tensor, state: torch.Tensor, is_final: bool = False):
        # x_new: [B, C, D], state: [B, S, D]
        buf = torch.cat([state, x_new], dim=1) if x_new.shape[1] > 0 else state
        if is_final and self.lookahead > 0:
            zeros_r = torch.zeros(buf.shape[0], self.lookahead, self.d_model, device=buf.device, dtype=buf.dtype)
            buf = torch.cat([buf, zeros_r], dim=1)
        if buf.shape[1] < self.kernel_size:
            empty = torch.zeros(buf.shape[0], 0, self.d_model, device=buf.device, dtype=buf.dtype)
            return empty, buf
        n_emit = buf.shape[1] - self.kernel_size + 1
        conv_out = self.dwconv(buf.transpose(1, 2))  # [B, D, n_emit]
        residual = buf[:, self.k_left : self.k_left + n_emit, :]
        out = self._apply_body(conv_out, residual)
        new_state = buf[:, n_emit:, :]
        return out, new_state


class CausalConv1d(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, kernel_size: int, groups: int = 1):
        super().__init__()
        self.kernel_size = kernel_size
        self.pad_left = kernel_size - 1
        self.in_ch = in_ch
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, groups=groups)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, C, T]
        return self.conv(F.pad(x, (self.pad_left, 0)))

    def init_state(self, batch_size: int, device, dtype):
        return torch.zeros(batch_size, self.in_ch, self.pad_left, device=device, dtype=dtype)

    def forward_step(self, x: torch.Tensor, state: torch.Tensor):
        # x: [B, C, T_chunk]
        if x.shape[-1] == 0:
            return torch.zeros(x.shape[0], self.conv.out_channels, 0, device=x.device, dtype=x.dtype), state
        buf = torch.cat([state, x], dim=-1)
        out = self.conv(buf)
        new_state = buf[:, :, -self.pad_left:] if self.pad_left > 0 else state
        return out, new_state


class CausalConvNeXtBlock(nn.Module):
    def __init__(self, d_model: int, kernel_size: int = 5, mult: int = 3):
        super().__init__()
        self.dwconv = CausalConv1d(d_model, d_model, kernel_size=kernel_size, groups=d_model)
        self.norm = nn.LayerNorm(d_model)
        self.pw1 = nn.Linear(d_model, d_model * mult)
        self.pw2 = nn.Linear(d_model * mult, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, T, D]
        h = self.dwconv(x.transpose(1, 2)).transpose(1, 2)
        h = self.pw2(F.gelu(self.pw1(self.norm(h))))
        return x + h

    def init_state(self, batch_size: int, device, dtype):
        return self.dwconv.init_state(batch_size, device, dtype)

    def forward_step(self, x: torch.Tensor, state: torch.Tensor):
        if x.shape[1] == 0:
            return x, state
        h, new_state = self.dwconv.forward_step(x.transpose(1, 2), state)
        h = self.pw2(F.gelu(self.pw1(self.norm(h.transpose(1, 2)))))
        return x + h, new_state


# ==============================================================================
# 5.2 五大序列主干（E0 离线双向 Transformer / E1 Conv+UniGRU / E2 SWA / E3 SSM / E4 RWKV）
# ==============================================================================
def apply_rope(x: torch.Tensor, pos: torch.Tensor) -> torch.Tensor:
    # x: [B, H, T, Dh], pos: [T]
    dh = x.shape[-1]
    inv_freq = 1.0 / (10000.0 ** (torch.arange(0, dh, 2, device=x.device, dtype=torch.float32) / dh))
    freqs = torch.outer(pos.to(torch.float32), inv_freq)  # [T, Dh/2]
    cos = freqs.cos()[None, None, :, :].to(x.dtype)
    sin = freqs.sin()[None, None, :, :].to(x.dtype)
    x1, x2 = x[..., 0::2], x[..., 1::2]
    out = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.flatten(-2)


class OfflineBiTransformerBackbone(nn.Module):
    """E0: 非流式全双向 Transformer（音质上界对照组）"""
    def __init__(self, d_model: int = 256, n_layers: int = 4, n_heads: int = 4, d_ff: int = 896):
        super().__init__()
        self.is_streaming = False
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=0.1, activation="gelu", batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers, enable_nested_tensor=False)
        self.pos_emb = nn.Embedding(512, d_model)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        T = x.shape[1]
        pos = torch.arange(T, device=x.device).clamp(max=511)
        h = x + self.pos_emb(pos)[None, :, :]
        pad_mask = (~mask) if mask is not None else None
        return self.encoder(h, src_key_padding_mask=pad_mask)


class ConvUniGRUBackbone(nn.Module):
    """E1: 集中右前瞻 + Causal ConvNeXt + 2 层单向 GRU（极简基线）"""
    def __init__(self, d_model: int = 256, k_left: int = 3, lookahead: int = 4, n_conv: int = 4):
        super().__init__()
        self.is_streaming = True
        self.lookahead_layer = CentralizedLookaheadBlock(d_model, k_left=k_left, lookahead=lookahead, mult=2)
        self.convs = nn.ModuleList([CausalConvNeXtBlock(d_model, kernel_size=5, mult=3) for _ in range(n_conv)])
        self.gru_norm = nn.LayerNorm(d_model)
        self.gru = nn.GRU(d_model, d_model, num_layers=2, batch_first=True, bidirectional=False)
        self.out_norm = nn.LayerNorm(d_model)
        self.out_ffn = nn.Sequential(nn.Linear(d_model, d_model * 3), nn.GELU(), nn.Linear(d_model * 3, d_model))

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        h = self.lookahead_layer(x)
        for blk in self.convs:
            h = blk(h)
        gru_out, _ = self.gru(self.gru_norm(h))
        h = h + gru_out
        return h + self.out_ffn(self.out_norm(h))

    def init_state(self, batch_size: int, device, dtype):
        return {
            "la": self.lookahead_layer.init_state(batch_size, device, dtype),
            "convs": [blk.init_state(batch_size, device, dtype) for blk in self.convs],
            "gru": torch.zeros(2, batch_size, self.gru.hidden_size, device=device, dtype=dtype),
        }

    def forward_step(self, x_new: torch.Tensor, state: dict, is_final: bool = False):
        h, la_st = self.lookahead_layer.forward_step(x_new, state["la"], is_final=is_final)
        if h.shape[1] == 0:
            return h, {"la": la_st, "convs": state["convs"], "gru": state["gru"]}
        new_convs = []
        for blk, cst in zip(self.convs, state["convs"]):
            h, nc = blk.forward_step(h, cst)
            new_convs.append(nc)
        gru_out, new_gru = self.gru(self.gru_norm(h), state["gru"])
        h = h + gru_out
        h = h + self.out_ffn(self.out_norm(h))
        return h, {"la": la_st, "convs": new_convs, "gru": new_gru}


class SWAConvLayer(nn.Module):
    """单层滑动窗口因果自注意力 (RoPE + 有界 KV Cache O(W)) + 因果 ConvNeXt"""
    def __init__(self, d_model: int = 256, n_heads: int = 4, window_size: int = 48):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.window_size = window_size
        self.norm1 = nn.LayerNorm(d_model)
        self.qkv = nn.Linear(d_model, d_model * 3)
        self.proj = nn.Linear(d_model, d_model)
        self.conv_ffn = CausalConvNeXtBlock(d_model, kernel_size=5, mult=2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, D = x.shape
        qkv = self.qkv(self.norm1(x)).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q = q.transpose(1, 2)  # [B, H, T, Dh]
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        pos = torch.arange(T, device=x.device)
        q = apply_rope(q, pos)
        k = apply_rope(k, pos)
        # 严格滑动窗口因果 Mask: 0 <= i - j < W
        diff = pos[:, None] - pos[None, :]
        allow = (diff >= 0) & (diff < self.window_size)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(~allow[None, None, :, :], float("-inf"))
        attn = torch.softmax(scores, dim=-1)
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, D)
        x = x + self.proj(out)
        return self.conv_ffn(x)

    def init_state(self, batch_size: int, device, dtype):
        return {
            "k": torch.zeros(batch_size, self.n_heads, 0, self.head_dim, device=device, dtype=dtype),
            "v": torch.zeros(batch_size, self.n_heads, 0, self.head_dim, device=device, dtype=dtype),
            "pos": 0,
            "conv": self.conv_ffn.init_state(batch_size, device, dtype),
        }

    def forward_step(self, x: torch.Tensor, state: dict):
        B, C, D = x.shape
        if C == 0:
            return x, state
        qkv = self.qkv(self.norm1(x)).view(B, C, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        pos_q = torch.arange(state["pos"], state["pos"] + C, device=x.device)
        q = apply_rope(q, pos_q)
        k = apply_rope(k, pos_q)
        k_all = torch.cat([state["k"], k], dim=2)
        v_all = torch.cat([state["v"], v], dim=2)
        k_len = k_all.shape[2]
        pos_k = torch.arange(state["pos"] + C - k_len, state["pos"] + C, device=x.device)
        diff = pos_q[:, None] - pos_k[None, :]
        allow = (diff >= 0) & (diff < self.window_size)
        scores = torch.matmul(q, k_all.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(~allow[None, None, :, :], float("-inf"))
        attn = torch.softmax(scores, dim=-1)
        out = torch.matmul(attn, v_all).transpose(1, 2).contiguous().view(B, C, D)
        x = x + self.proj(out)
        x, new_conv = self.conv_ffn.forward_step(x, state["conv"])
        keep = self.window_size - 1
        return x, {
            "k": k_all[:, :, -keep:, :] if keep > 0 else k_all[:, :, :0, :],
            "v": v_all[:, :, -keep:, :] if keep > 0 else v_all[:, :, :0, :],
            "pos": state["pos"] + C,
            "conv": new_conv,
        }


class SWABackbone(nn.Module):
    """E2: 集中右前瞻 + 5 层滑动窗口因果注意力 SWA"""
    def __init__(self, d_model: int = 256, k_left: int = 3, lookahead: int = 4, n_layers: int = 5, n_heads: int = 4, window_size: int = 48):
        super().__init__()
        self.is_streaming = True
        self.lookahead_layer = CentralizedLookaheadBlock(d_model, k_left=k_left, lookahead=lookahead, mult=2)
        self.layers = nn.ModuleList([SWAConvLayer(d_model, n_heads=n_heads, window_size=window_size) for _ in range(n_layers)])

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        h = self.lookahead_layer(x)
        for layer in self.layers:
            h = layer(h)
        return h

    def init_state(self, batch_size: int, device, dtype):
        return {
            "la": self.lookahead_layer.init_state(batch_size, device, dtype),
            "layers": [l.init_state(batch_size, device, dtype) for l in self.layers],
        }

    def forward_step(self, x_new: torch.Tensor, state: dict, is_final: bool = False):
        h, la_st = self.lookahead_layer.forward_step(x_new, state["la"], is_final=is_final)
        if h.shape[1] == 0:
            return h, {"la": la_st, "layers": state["layers"]}
        new_layers = []
        for l, st in zip(self.layers, state["layers"]):
            h, nst = l.forward_step(h, st)
            new_layers.append(nst)
        return h, {"la": la_st, "layers": new_layers}


class SelectiveSSMBlock(nn.Module):
    """Mamba 风格纯 PyTorch Selective SSM 块（无 CUDA 编译依赖，训练与 O(1) 流式严格等价）"""
    def __init__(self, d_model: int = 256, expand: int = 2, d_state: int = 16, d_conv: int = 4):
        super().__init__()
        self.d_model = d_model
        self.d_inner = d_model * expand
        self.d_state = d_state
        self.dt_rank = max(16, d_model // 16)

        self.norm = nn.LayerNorm(d_model)
        self.in_proj = nn.Linear(d_model, self.d_inner * 2)
        self.conv = CausalConv1d(self.d_inner, self.d_inner, kernel_size=d_conv, groups=self.d_inner)
        self.x_proj = nn.Linear(self.d_inner, self.dt_rank + 2 * d_state, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True)

        A = torch.arange(1, d_state + 1, dtype=torch.float32).unsqueeze(0).repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D_skip = nn.Parameter(torch.ones(self.d_inner))
        self.out_proj = nn.Linear(self.d_inner, d_model)

    def _scan(self, x_silu: torch.Tensor, h_init: torch.Tensor):
        # x_silu: [B, T, d_inner], h_init: [B, d_inner, d_state] (float32)
        B_sz, T, D_in = x_silu.shape
        proj = self.x_proj(x_silu)
        dt_raw, B_mat, C_mat = torch.split(proj, [self.dt_rank, self.d_state, self.d_state], dim=-1)
        dt = F.softplus(self.dt_proj(dt_raw)).float()  # [B, T, d_inner]
        A = -torch.exp(self.A_log.float())  # [d_inner, d_state]
        B_f = B_mat.float()
        C_f = C_mat.float()
        x_f = x_silu.float()

        h = h_init.float()
        ys = []
        for t in range(T):
            dt_t = dt[:, t, :].unsqueeze(-1)              # [B, d_inner, 1]
            dA_t = torch.exp(dt_t * A.unsqueeze(0))       # [B, d_inner, d_state]
            dBx_t = (dt_t * x_f[:, t, :].unsqueeze(-1)) * B_f[:, t, :].unsqueeze(1)
            h = dA_t * h + dBx_t
            y_t = torch.sum(h * C_f[:, t, :].unsqueeze(1), dim=-1) + self.D_skip.float() * x_f[:, t, :]
            ys.append(y_t)
        y = torch.stack(ys, dim=1).to(x_silu.dtype)
        return y, h

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B_sz = x.shape[0]
        xz = self.in_proj(self.norm(x))
        x_b, z = xz.chunk(2, dim=-1)
        x_c = F.silu(self.conv(x_b.transpose(1, 2)).transpose(1, 2))
        h0 = torch.zeros(B_sz, self.d_inner, self.d_state, device=x.device, dtype=torch.float32)
        y, _ = self._scan(x_c, h0)
        return x + self.out_proj(y * F.silu(z))

    def init_state(self, batch_size: int, device, dtype):
        return {
            "conv": self.conv.init_state(batch_size, device, dtype),
            "h": torch.zeros(batch_size, self.d_inner, self.d_state, device=device, dtype=torch.float32),
        }

    def forward_step(self, x: torch.Tensor, state: dict):
        if x.shape[1] == 0:
            return x, state
        xz = self.in_proj(self.norm(x))
        x_b, z = xz.chunk(2, dim=-1)
        x_c, new_conv = self.conv.forward_step(x_b.transpose(1, 2), state["conv"])
        x_c = F.silu(x_c.transpose(1, 2))
        y, new_h = self._scan(x_c, state["h"])
        return x + self.out_proj(y * F.silu(z)), {"conv": new_conv, "h": new_h}


class SelectiveSSMBackbone(nn.Module):
    """E3: 集中右前瞻 + 5 层 Selective SSM (Mamba 风格)"""
    def __init__(self, d_model: int = 256, k_left: int = 3, lookahead: int = 4, n_layers: int = 5):
        super().__init__()
        self.is_streaming = True
        self.lookahead_layer = CentralizedLookaheadBlock(d_model, k_left=k_left, lookahead=lookahead, mult=2)
        self.layers = nn.ModuleList([SelectiveSSMBlock(d_model, expand=2, d_state=16, d_conv=4) for _ in range(n_layers)])

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        h = self.lookahead_layer(x)
        for l in self.layers:
            h = l(h)
        return h

    def init_state(self, batch_size: int, device, dtype):
        return {
            "la": self.lookahead_layer.init_state(batch_size, device, dtype),
            "layers": [l.init_state(batch_size, device, dtype) for l in self.layers],
        }

    def forward_step(self, x_new: torch.Tensor, state: dict, is_final: bool = False):
        h, la_st = self.lookahead_layer.forward_step(x_new, state["la"], is_final=is_final)
        if h.shape[1] == 0:
            return h, {"la": la_st, "layers": state["layers"]}
        new_layers = []
        for l, st in zip(self.layers, state["layers"]):
            h, nst = l.forward_step(h, st)
            new_layers.append(nst)
        return h, {"la": la_st, "layers": new_layers}


class RWKVBlock(nn.Module):
    """RWKV-v6 风格纯 PyTorch 数据依赖线性注意力 WKV 块（O(1) 矩阵状态，极适配 ARM CPU）"""
    def __init__(self, d_model: int = 256, n_heads: int = 4, ffn_mult: int = 3):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        # Time-Mixing
        self.norm1 = nn.LayerNorm(d_model)
        self.mu_r = nn.Parameter(torch.full((1, 1, d_model), 0.5))
        self.mu_k = nn.Parameter(torch.full((1, 1, d_model), 0.5))
        self.mu_v = nn.Parameter(torch.full((1, 1, d_model), 0.5))
        self.mu_w = nn.Parameter(torch.full((1, 1, d_model), 0.5))
        self.mu_g = nn.Parameter(torch.full((1, 1, d_model), 0.5))

        self.receptance = nn.Linear(d_model, d_model, bias=False)
        self.key = nn.Linear(d_model, d_model, bias=False)
        self.value = nn.Linear(d_model, d_model, bias=False)
        self.gate = nn.Linear(d_model, d_model, bias=False)
        self.w_base = nn.Parameter(torch.full((n_heads, self.head_dim), -1.0))
        self.w_lora1 = nn.Linear(d_model, 32, bias=False)
        self.w_lora2 = nn.Linear(32, d_model, bias=True)
        self.u_bonus = nn.Parameter(torch.zeros(n_heads, self.head_dim))
        self.head_norm = nn.GroupNorm(n_heads, d_model)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        # Channel-Mixing
        self.norm2 = nn.LayerNorm(d_model)
        self.mu_ck = nn.Parameter(torch.full((1, 1, d_model), 0.5))
        self.mu_cr = nn.Parameter(torch.full((1, 1, d_model), 0.5))
        self.ck = nn.Linear(d_model, d_model * ffn_mult, bias=False)
        self.cv = nn.Linear(d_model * ffn_mult, d_model, bias=False)
        self.cr = nn.Linear(d_model, d_model, bias=False)

    def _run(self, x: torch.Tensor, last_t: torch.Tensor, wkv_s: torch.Tensor, last_c: torch.Tensor):
        B, T, D = x.shape
        xn = self.norm1(x)
        prev_t = torch.cat([last_t, xn[:, :-1, :]], dim=1)
        xr = xn * self.mu_r + prev_t * (1.0 - self.mu_r)
        xk = xn * self.mu_k + prev_t * (1.0 - self.mu_k)
        xv = xn * self.mu_v + prev_t * (1.0 - self.mu_v)
        xw = xn * self.mu_w + prev_t * (1.0 - self.mu_w)
        xg = xn * self.mu_g + prev_t * (1.0 - self.mu_g)

        r = self.receptance(xr).view(B, T, self.n_heads, self.head_dim).float()
        k = self.key(xk).view(B, T, self.n_heads, self.head_dim).float()
        v = self.value(xv).view(B, T, self.n_heads, self.head_dim).float()
        w_dyn = self.w_lora2(torch.tanh(self.w_lora1(xw))).view(B, T, self.n_heads, self.head_dim).float()
        decay = torch.exp(-torch.exp(self.w_base[None, None, :, :].float() + w_dyn).clamp(max=10.0))  # in (0, 1)
        g = F.silu(self.gate(xg))

        S = wkv_s.float()  # [B, H, Dh, Dh]
        u = self.u_bonus[None, :, :].float()
        ys = []
        for t in range(T):
            rt, kt, vt, wt = r[:, t], k[:, t], v[:, t], decay[:, t]
            # y_t: [B, H, Dh]
            yt = torch.einsum("bhd,bhde->bhe", rt, S) + (rt * u * kt).sum(dim=-1, keepdim=True) * vt
            S = wt.unsqueeze(-1) * S + torch.einsum("bhd,bhe->bhde", kt, vt)
            ys.append(yt)
        y = torch.stack(ys, dim=1).reshape(B * T, D).to(x.dtype)
        y = self.head_norm(y).reshape(B, T, D)
        x = x + self.out_proj(y * g)
        new_last_t = xn[:, -1:, :]

        # Channel-Mixing
        xc = self.norm2(x)
        prev_c = torch.cat([last_c, xc[:, :-1, :]], dim=1)
        xck = xc * self.mu_ck + prev_c * (1.0 - self.mu_ck)
        xcr = xc * self.mu_cr + prev_c * (1.0 - self.mu_cr)
        cm = torch.sigmoid(self.cr(xcr)) * self.cv(F.relu(self.ck(xck)) ** 2)
        x = x + cm
        new_last_c = xc[:, -1:, :]
        return x, new_last_t, S, new_last_c

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B = x.shape[0]
        z1 = torch.zeros(B, 1, self.d_model, device=x.device, dtype=x.dtype)
        S0 = torch.zeros(B, self.n_heads, self.head_dim, self.head_dim, device=x.device, dtype=torch.float32)
        out, _, _, _ = self._run(x, z1, S0, z1)
        return out

    def init_state(self, batch_size: int, device, dtype):
        return {
            "last_t": torch.zeros(batch_size, 1, self.d_model, device=device, dtype=dtype),
            "S": torch.zeros(batch_size, self.n_heads, self.head_dim, self.head_dim, device=device, dtype=torch.float32),
            "last_c": torch.zeros(batch_size, 1, self.d_model, device=device, dtype=dtype),
        }

    def forward_step(self, x: torch.Tensor, state: dict):
        if x.shape[1] == 0:
            return x, state
        out, lt, S, lc = self._run(x, state["last_t"], state["S"], state["last_c"])
        return out, {"last_t": lt, "S": S, "last_c": lc}


class RWKVBackbone(nn.Module):
    """E4: 集中右前瞻 + 5 层 RWKV-v6 风格线性注意力"""
    def __init__(self, d_model: int = 256, k_left: int = 3, lookahead: int = 4, n_layers: int = 5, n_heads: int = 4):
        super().__init__()
        self.is_streaming = True
        self.lookahead_layer = CentralizedLookaheadBlock(d_model, k_left=k_left, lookahead=lookahead, mult=2)
        self.layers = nn.ModuleList([RWKVBlock(d_model, n_heads=n_heads, ffn_mult=2) for _ in range(n_layers)])

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        h = self.lookahead_layer(x)
        for l in self.layers:
            h = l(h)
        return h

    def init_state(self, batch_size: int, device, dtype):
        return {
            "la": self.lookahead_layer.init_state(batch_size, device, dtype),
            "layers": [l.init_state(batch_size, device, dtype) for l in self.layers],
        }

    def forward_step(self, x_new: torch.Tensor, state: dict, is_final: bool = False):
        h, la_st = self.lookahead_layer.forward_step(x_new, state["la"], is_final=is_final)
        if h.shape[1] == 0:
            return h, {"la": la_st, "layers": state["layers"]}
        new_layers = []
        for l, st in zip(self.layers, state["layers"]):
            h, nst = l.forward_step(h, st)
            new_layers.append(nst)
        return h, {"la": la_st, "layers": new_layers}


# ==============================================================================
# 5.3 内置单调对齐器 (RadTTS/FastPitch 风格，训练自动学出精确音素时长，免 MFA 依赖)
# ==============================================================================
class AlignmentEncoder(nn.Module):
    def __init__(self, n_mels: int = 100, d_text: int = 256, d_attn: int = 80):
        super().__init__()
        self.key_proj = nn.Sequential(
            nn.Conv1d(d_text, d_text, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(d_text, d_attn, kernel_size=1),
        )
        self.query_proj = nn.Sequential(
            nn.Conv1d(n_mels, n_mels * 2, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(n_mels * 2, d_attn, kernel_size=1),
        )
        self.scale = d_attn ** -0.5

    def forward(self, mel: torch.Tensor, text_enc: torch.Tensor, text_mask: torch.Tensor):
        # mel: [B, n_mels, F], text_enc: [B, T, D], text_mask: [B, T] (True=valid)
        keys = self.key_proj(text_enc.transpose(1, 2))        # [B, d_attn, T]
        queries = self.query_proj(mel)                        # [B, d_attn, F]
        # 负 L2 距离相似度（比点积更易收敛出清晰单调对角线）
        dist = torch.cdist(queries.transpose(1, 2), keys.transpose(1, 2), p=2.0)  # [B, F, T]
        log_prob = -0.5 * (dist ** 2) * self.scale
        log_prob = log_prob.masked_fill(~text_mask[:, None, :], -1e4)
        attn_logprob = F.log_softmax(log_prob, dim=-1)        # [B, F, T]
        attn_soft = torch.softmax(log_prob, dim=-1)           # [B, F, T]
        return attn_logprob, attn_soft


def forward_sum_loss(attn_logprob: torch.Tensor, text_lens: torch.Tensor, mel_lens: torch.Tensor) -> torch.Tensor:
    """利用 PyTorch 原生 CTCLoss 实现高效 Forward-Sum 单调对齐损失"""
    B, F_max, T_max = attn_logprob.shape
    # 为 CTC blank 添加一列极小概率 (index 0 = blank, 1..T_max = phoneme indices)
    blank = torch.full((B, F_max, 1), -1e4, device=attn_logprob.device, dtype=attn_logprob.dtype)
    log_probs = torch.cat([blank, attn_logprob], dim=-1).transpose(0, 1)  # [F_max, B, T_max+1]
    targets = torch.cat([torch.arange(1, tl + 1, device=attn_logprob.device, dtype=torch.long) for tl in text_lens])
    return F.ctc_loss(log_probs.float(), targets, mel_lens, text_lens, blank=0, zero_infinity=True)


@torch.no_grad()
def viterbi_monotonic_alignment(attn_logprob: torch.Tensor, text_lens: torch.Tensor, mel_lens: torch.Tensor) -> torch.Tensor:
    """纯 NumPy Viterbi 单调对齐搜索 (MAS)，返回每个音素的整数时长 [B, T_max]"""
    B, F_max, T_max = attn_logprob.shape
    lp_np = attn_logprob.detach().cpu().float().numpy()
    durations = np.zeros((B, T_max), dtype=np.int64)
    for b in range(B):
        F_b = int(mel_lens[b].item())
        T_b = int(text_lens[b].item())
        val = lp_np[b, :F_b, :T_b]  # [F_b, T_b]
        dp = np.full((F_b, T_b), -1e9, dtype=np.float32)
        dp[0, 0] = val[0, 0]
        for f in range(1, F_b):
            dp[f, 0] = dp[f - 1, 0] + val[f, 0]
            t_max = min(f + 1, T_b)
            prev_stay = dp[f - 1, 1:t_max]
            prev_step = dp[f - 1, 0:t_max - 1]
            dp[f, 1:t_max] = np.maximum(prev_stay, prev_step) + val[f, 1:t_max]
        # 回溯
        t = T_b - 1
        for f in range(F_b - 1, -1, -1):
            durations[b, t] += 1
            if f > 0 and t > 0 and dp[f - 1, t - 1] >= dp[f - 1, t]:
                t -= 1
    return torch.from_numpy(durations).to(attn_logprob.device)


def expand_by_duration(h: torch.Tensor, durations: torch.Tensor, max_frames: int = None):
    """Length Regulator: 按每音素时长展开 [B, T, D] -> [B, F, D]"""
    B, T, D = h.shape
    outs = []
    out_lens = []
    for b in range(B):
        d = durations[b].clamp(min=0)
        rep = torch.repeat_interleave(h[b], d, dim=0)
        if max_frames is not None:
            if rep.shape[0] < max_frames:
                rep = F.pad(rep, (0, 0, 0, max_frames - rep.shape[0]))
            else:
                rep = rep[:max_frames]
        outs.append(rep)
        out_lens.append(rep.shape[0])
    F_max = max(out_lens) if max_frames is None else max_frames
    padded = torch.zeros(B, F_max, D, device=h.device, dtype=h.dtype)
    for b, rep in enumerate(outs):
        padded[b, :rep.shape[0]] = rep
    return padded, torch.tensor(out_lens, device=h.device, dtype=torch.long)


# ==============================================================================
# 5.4 完整声学模型（统一包裹 E0-E4，显式 Duration / F0 / Energy + 因果帧解码）
# ==============================================================================
class CausalPredictorHead(nn.Module):
    def __init__(self, d_model: int = 256, kernel_size: int = 3):
        super().__init__()
        self.c1 = CausalConv1d(d_model, d_model, kernel_size)
        self.n1 = nn.LayerNorm(d_model)
        self.c2 = CausalConv1d(d_model, d_model, kernel_size)
        self.n2 = nn.LayerNorm(d_model)
        self.proj = nn.Linear(d_model, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = F.silu(self.n1(self.c1(x.transpose(1, 2)).transpose(1, 2)))
        h = F.silu(self.n2(self.c2(h.transpose(1, 2)).transpose(1, 2)))
        return self.proj(h).squeeze(-1)

    def init_state(self, batch_size: int, device, dtype):
        return (self.c1.init_state(batch_size, device, dtype), self.c2.init_state(batch_size, device, dtype))

    def forward_step(self, x: torch.Tensor, state):
        if x.shape[1] == 0:
            return torch.zeros(x.shape[0], 0, device=x.device, dtype=x.dtype), state
        h, s1 = self.c1.forward_step(x.transpose(1, 2), state[0])
        h = F.silu(self.n1(h.transpose(1, 2)))
        h, s2 = self.c2.forward_step(h.transpose(1, 2), state[1])
        h = F.silu(self.n2(h.transpose(1, 2)))
        return self.proj(h).squeeze(-1), (s1, s2)


class AetherAcousticModel(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        backbone_type: str = "swa",
        d_model: int = 256,
        n_mels: int = 100,
        k_left: int = 3,
        lookahead: int = 4,
        swa_window: int = 48,
    ):
        super().__init__()
        self.backbone_type = backbone_type
        self.d_model = d_model
        self.n_mels = n_mels
        self.lookahead = lookahead

        self.phoneme_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.lang_emb = nn.Embedding(4, d_model)

        if backbone_type == "offline_transformer":
            self.backbone = OfflineBiTransformerBackbone(d_model=d_model, n_layers=4, n_heads=4, d_ff=896)
        elif backbone_type == "conv_gru":
            self.backbone = ConvUniGRUBackbone(d_model=d_model, k_left=k_left, lookahead=lookahead, n_conv=4)
        elif backbone_type == "swa":
            self.backbone = SWABackbone(d_model=d_model, k_left=k_left, lookahead=lookahead, n_layers=5, n_heads=4, window_size=swa_window)
        elif backbone_type == "ssm":
            self.backbone = SelectiveSSMBackbone(d_model=d_model, k_left=k_left, lookahead=lookahead, n_layers=5)
        elif backbone_type == "rwkv":
            self.backbone = RWKVBackbone(d_model=d_model, k_left=k_left, lookahead=lookahead, n_layers=5, n_heads=4)
        else:
            raise ValueError(f"Unknown backbone_type: {backbone_type}")

        self.is_streaming = self.backbone.is_streaming
        self.aligner = AlignmentEncoder(n_mels=n_mels, d_text=d_model, d_attn=80)
        self.dur_predictor = CausalPredictorHead(d_model, kernel_size=3)
        self.f0_predictor = CausalPredictorHead(d_model, kernel_size=3)
        self.energy_predictor = CausalPredictorHead(d_model, kernel_size=3)
        self.f0_emb = nn.Linear(1, d_model)
        self.energy_emb = nn.Linear(1, d_model)

        # 帧级因果解码器（4 层 Causal ConvNeXt -> 100-bin Log-Mel）
        self.frame_decoder = nn.ModuleList([CausalConvNeXtBlock(d_model, kernel_size=5, mult=2) for _ in range(4)])
        self.mel_out = nn.Linear(d_model, n_mels)

    def encode_text(self, phoneme_ids: torch.Tensor, lang_ids: torch.Tensor, text_mask: torch.Tensor = None):
        x = self.phoneme_emb(phoneme_ids) + self.lang_emb(lang_ids)
        return self.backbone(x, mask=text_mask)

    def forward_train(self, phoneme_ids, lang_ids, text_lens, mel_gt, f0_gt, energy_gt, mel_lens):
        B, T_max = phoneme_ids.shape
        F_max = mel_gt.shape[-1]
        text_mask = torch.arange(T_max, device=phoneme_ids.device)[None, :] < text_lens[:, None]
        mel_mask = torch.arange(F_max, device=phoneme_ids.device)[None, :] < mel_lens[:, None]

        h_txt = self.encode_text(phoneme_ids, lang_ids, text_mask)
        # 1) 单调对齐与目标时长
        attn_logprob, attn_soft = self.aligner(mel_gt, h_txt, text_mask)
        loss_align = forward_sum_loss(attn_logprob, text_lens, mel_lens)
        dur_gt = viterbi_monotonic_alignment(attn_logprob, text_lens, mel_lens)  # [B, T_max]

        # 2) 时长预测损失 (log(dur + 1))
        log_dur_pred = self.dur_predictor(h_txt)
        log_dur_gt = torch.log(dur_gt.float() + 1.0)
        loss_dur = (F.l1_loss(log_dur_pred, log_dur_gt, reduction="none") * text_mask).sum() / text_mask.sum().clamp(min=1)

        # 3) 按目标时长展开 -> F0 / Energy 预测与注入
        h_frame, _ = expand_by_duration(h_txt, dur_gt, max_frames=F_max)
        f0_pred = self.f0_predictor(h_frame)
        energy_pred = self.energy_predictor(h_frame)
        loss_f0 = (F.l1_loss(f0_pred, f0_gt, reduction="none") * mel_mask).sum() / mel_mask.sum().clamp(min=1)
        loss_energy = (F.l1_loss(energy_pred, energy_gt, reduction="none") * mel_mask).sum() / mel_mask.sum().clamp(min=1)

        # 训练期 Teacher-Forcing 注入 F0/Energy（50% 混合预测值缓解暴露偏差）
        use_pred = (torch.rand(1).item() < 0.25)
        f0_in = f0_pred.detach() if use_pred else f0_gt
        e_in = energy_pred.detach() if use_pred else energy_gt
        h_dec = h_frame + self.f0_emb(f0_in.unsqueeze(-1)) + self.energy_emb(e_in.unsqueeze(-1))
        for blk in self.frame_decoder:
            h_dec = blk(h_dec)
        mel_pred = self.mel_out(h_dec).transpose(1, 2)  # [B, n_mels, F_max]

        loss_mel = (F.l1_loss(mel_pred, mel_gt, reduction="none") * mel_mask[:, None, :]).sum() / (mel_mask.sum().clamp(min=1) * self.n_mels)
        total_loss = loss_mel + 0.5 * loss_dur + 0.25 * loss_f0 + 0.25 * loss_energy + 0.2 * loss_align
        return {
            "loss": total_loss,
            "mel": loss_mel.detach(),
            "dur": loss_dur.detach(),
            "f0": loss_f0.detach(),
            "energy": loss_energy.detach(),
            "align": loss_align.detach(),
            "mel_pred": mel_pred.detach(),
        }

    @torch.no_grad()
    def infer_offline(self, phoneme_ids: torch.Tensor, lang_ids: torch.Tensor, speed: float = 1.0):
        # phoneme_ids: [1, T]
        h_txt = self.encode_text(phoneme_ids, lang_ids)
        log_dur = self.dur_predictor(h_txt)
        dur = torch.clamp(torch.round((torch.exp(log_dur) - 1.0) / speed), min=1, max=40).long()
        h_frame, _ = expand_by_duration(h_txt, dur)
        f0_pred = self.f0_predictor(h_frame)
        energy_pred = self.energy_predictor(h_frame)
        h_dec = h_frame + self.f0_emb(f0_pred.unsqueeze(-1)) + self.energy_emb(energy_pred.unsqueeze(-1))
        for blk in self.frame_decoder:
            h_dec = blk(h_dec)
        mel = self.mel_out(h_dec).transpose(1, 2)  # [1, n_mels, F]
        return mel, dur, f0_pred, energy_pred

    def init_stream_state(self, batch_size: int = 1, device=None, dtype=torch.float32):
        assert self.is_streaming, f"{self.backbone_type} is non-streaming!"
        device = device or next(self.parameters()).device
        return {
            "backbone": self.backbone.init_state(batch_size, device, dtype),
            "dur": self.dur_predictor.init_state(batch_size, device, dtype),
            "f0": self.f0_predictor.init_state(batch_size, device, dtype),
            "energy": self.energy_predictor.init_state(batch_size, device, dtype),
            "frame_dec": [blk.init_state(batch_size, device, dtype) for blk in self.frame_decoder],
        }

    @torch.no_grad()
    def infer_stream_step(self, phoneme_chunk: torch.Tensor, lang_chunk: torch.Tensor, state: dict, is_final: bool = False, speed: float = 1.0):
        """真正的流式输入→流式输出步进：喂入音素块，立即吐出已满足右前瞻条件对应的梅尔帧块！"""
        if phoneme_chunk.shape[1] > 0:
            x = self.phoneme_emb(phoneme_chunk) + self.lang_emb(lang_chunk)
        else:
            x = torch.zeros(phoneme_chunk.shape[0], 0, self.d_model, device=phoneme_chunk.device, dtype=self.phoneme_emb.weight.dtype)

        h_emit, st_bb = self.backbone.forward_step(x, state["backbone"], is_final=is_final)
        state["backbone"] = st_bb
        if h_emit.shape[1] == 0:
            return torch.zeros(x.shape[0], self.n_mels, 0, device=x.device, dtype=x.dtype), state

        log_dur, state["dur"] = self.dur_predictor.forward_step(h_emit, state["dur"])
        dur = torch.clamp(torch.round((torch.exp(log_dur) - 1.0) / speed), min=1, max=40).long()
        h_frame, _ = expand_by_duration(h_emit, dur)
        f0_pred, state["f0"] = self.f0_predictor.forward_step(h_frame, state["f0"])
        e_pred, state["energy"] = self.energy_predictor.forward_step(h_frame, state["energy"])
        h_dec = h_frame + self.f0_emb(f0_pred.unsqueeze(-1)) + self.energy_emb(e_pred.unsqueeze(-1))
        new_fdec = []
        for blk, fst in zip(self.frame_decoder, state["frame_dec"]):
            h_dec, nfst = blk.forward_step(h_dec, fst)
            new_fdec.append(nfst)
        state["frame_dec"] = new_fdec
        mel_chunk = self.mel_out(h_dec).transpose(1, 2)  # [B, n_mels, F_chunk]
        return mel_chunk, state


# ==============================================================================
# 5.5 自研 24 kHz 因果流式声码器 CausalVocosLite (~3.5M) + 零爆音 OLA iSTFT + 判别器
# ==============================================================================
class StreamingISTFTHead(nn.Module):
    """
    将隐状态映射为复数谱 (mag, phase) 并执行精确 Overlap-Add iSTFT。
    整句 forward() 与流式 forward_step() 数学完全等价（0 爆音）。
    """
    def __init__(self, dim: int, n_fft: int = 1024, hop_length: int = 256):
        super().__init__()
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.n_bins = n_fft // 2 + 1
        self.out = nn.Linear(dim, n_fft + 2)
        self.register_buffer("window", torch.hann_window(n_fft, periodic=True))

    def _frames_to_time(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, F, dim] -> time frames [B, F, n_fft]
        pred = self.out(x).float()
        mag, p = pred.chunk(2, dim=-1)
        mag = torch.exp(mag).clamp(max=1e2)
        cos_p = torch.cos(p)
        sin_p = torch.sin(p)
        spec = torch.complex(mag * cos_p, mag * sin_p)  # [B, F, n_bins]
        frames = torch.fft.irfft(spec, n=self.n_fft, dim=-1) * self.window[None, None, :]
        return frames

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, F, dim] -> wav: [B, F * hop_length]
        B, n_frames, _ = x.shape
        frames = self._frames_to_time(x)  # [B, F, n_fft]
        pad = self.n_fft - self.hop_length
        total_len = n_frames * self.hop_length + pad
        out = torch.zeros(B, total_len, device=x.device, dtype=torch.float32)
        env = torch.zeros(B, total_len, device=x.device, dtype=torch.float32)
        w2 = (self.window ** 2)[None, :]
        for m in range(n_frames):
            st = m * self.hop_length
            out[:, st : st + self.n_fft] += frames[:, m, :]
            env[:, st : st + self.n_fft] += w2
        # 居中对齐截取，恰好输出 n_frames * hop_length 采样点
        left_trim = pad // 2
        out = out[:, left_trim : left_trim + n_frames * self.hop_length]
        env = env[:, left_trim : left_trim + n_frames * self.hop_length]
        return out / env.clamp(min=1e-6)

    def init_state(self, batch_size: int, device):
        pad = self.n_fft - self.hop_length
        return {
            "ola_out": torch.zeros(batch_size, pad, device=device, dtype=torch.float32),
            "ola_env": torch.zeros(batch_size, pad, device=device, dtype=torch.float32),
            "trim_remaining": pad // 2,
        }

    @torch.no_grad()
    def forward_step(self, x_chunk: torch.Tensor, state: dict, is_final: bool = False):
        # x_chunk: [B, C, dim]
        B, C, _ = x_chunk.shape
        pad = self.n_fft - self.hop_length
        left_trim = pad // 2  # 384 samples
        if C == 0 and not is_final:
            return torch.zeros(B, 0, device=x_chunk.device, dtype=torch.float32), state

        out_len = C * self.hop_length + pad
        out = torch.zeros(B, out_len, device=x_chunk.device, dtype=torch.float32)
        env = torch.zeros(B, out_len, device=x_chunk.device, dtype=torch.float32)
        out[:, :pad] = state["ola_out"]
        env[:, :pad] = state["ola_env"]

        if C > 0:
            frames = self._frames_to_time(x_chunk)
            w2 = (self.window ** 2)[None, :]
            for m in range(C):
                st = m * self.hop_length
                out[:, st : st + self.n_fft] += frames[:, m, :]
                env[:, st : st + self.n_fft] += w2

        emit_end = C * self.hop_length
        trim_rem = state["trim_remaining"]

        if is_final:
            tail_end = emit_end + (pad - left_trim)
            if tail_end <= trim_rem:
                wav_emit = torch.zeros(B, 0, device=x_chunk.device, dtype=torch.float32)
            else:
                wav_emit = out[:, trim_rem:tail_end] / env[:, trim_rem:tail_end].clamp(min=1e-6)
            return wav_emit, self.init_state(B, x_chunk.device)

        if emit_end <= trim_rem:
            wav_emit = torch.zeros(B, 0, device=x_chunk.device, dtype=torch.float32)
            new_trim = trim_rem - emit_end
        else:
            wav_emit = out[:, trim_rem:emit_end] / env[:, trim_rem:emit_end].clamp(min=1e-6)
            new_trim = 0

        new_state = {
            "ola_out": out[:, emit_end : emit_end + pad].clone(),
            "ola_env": env[:, emit_end : emit_end + pad].clone(),
            "trim_remaining": new_trim,
        }
        return wav_emit, new_state


class CausalVocosLite(nn.Module):
    """自研 ~3.5M 24 kHz 严格因果流式声码器（适配嵌入式 ARM CPU）"""
    def __init__(self, n_mels: int = 100, dim: int = 256, n_layers: int = 8, kernel_size: int = 7, n_fft: int = 1024, hop_length: int = 256):
        super().__init__()
        self.embed = CausalConv1d(n_mels, dim, kernel_size=7)
        self.norm_in = nn.LayerNorm(dim)
        self.blocks = nn.ModuleList([CausalConvNeXtBlock(dim, kernel_size=kernel_size, mult=3) for _ in range(n_layers)])
        self.norm_out = nn.LayerNorm(dim)
        self.head = StreamingISTFTHead(dim, n_fft=n_fft, hop_length=hop_length)

    def forward(self, mel: torch.Tensor) -> torch.Tensor:
        # mel: [B, n_mels, F] -> wav: [B, F * hop_length]
        h = self.norm_in(self.embed(mel).transpose(1, 2))
        for blk in self.blocks:
            h = blk(h)
        h = self.norm_out(h)
        return self.head(h)

    def init_state(self, batch_size: int = 1, device=None, dtype=torch.float32):
        device = device or next(self.parameters()).device
        return {
            "embed": self.embed.init_state(batch_size, device, dtype),
            "blocks": [blk.init_state(batch_size, device, dtype) for blk in self.blocks],
            "head": self.head.init_state(batch_size, device),
        }

    @torch.no_grad()
    def forward_step(self, mel_chunk: torch.Tensor, state: dict, is_final: bool = False):
        # mel_chunk: [B, n_mels, C]
        if mel_chunk.shape[-1] == 0:
            h = torch.zeros(mel_chunk.shape[0], 0, self.norm_out.normalized_shape[0], device=mel_chunk.device, dtype=mel_chunk.dtype)
            wav, state["head"] = self.head.forward_step(h, state["head"], is_final=is_final)
            return wav, state
        h, state["embed"] = self.embed.forward_step(mel_chunk, state["embed"])
        h = self.norm_in(h.transpose(1, 2))
        new_blks = []
        for blk, bst in zip(self.blocks, state["blocks"]):
            h, nb = blk.forward_step(h, bst)
            new_blks.append(nb)
        state["blocks"] = new_blks
        h = self.norm_out(h)
        wav, state["head"] = self.head.forward_step(h, state["head"], is_final=is_final)
        return wav, state


# 轻量多尺度频谱 + 多周期判别器（仅训练期启用，落实 Paradee 防嗡嗡声对抗配方）
class PeriodDisc(nn.Module):
    def __init__(self, period: int):
        super().__init__()
        self.period = period
        self.convs = nn.ModuleList([
            nn.Conv2d(1, 32, (5, 1), (3, 1), (2, 0)),
            nn.Conv2d(32, 64, (5, 1), (3, 1), (2, 0)),
            nn.Conv2d(64, 128, (5, 1), (3, 1), (2, 0)),
            nn.Conv2d(128, 256, (5, 1), (3, 1), (2, 0)),
            nn.Conv2d(256, 1, (3, 1), 1, (1, 0)),
        ])

    def forward(self, x: torch.Tensor):
        B, T = x.shape
        if T % self.period != 0:
            x = F.pad(x, (0, self.period - T % self.period), mode="reflect")
        x = x.view(B, 1, -1, self.period)
        fmap = []
        for i, conv in enumerate(self.convs):
            x = conv(x)
            if i < len(self.convs) - 1:
                x = F.leaky_relu(x, 0.1)
            fmap.append(x)
        return x.flatten(1, -1), fmap


class SpecDisc(nn.Module):
    def __init__(self, n_fft: int, hop: int):
        super().__init__()
        self.n_fft = n_fft
        self.hop = hop
        self.register_buffer("win", torch.hann_window(n_fft))
        self.convs = nn.ModuleList([
            nn.Conv2d(1, 32, (3, 9), (1, 1), (1, 4)),
            nn.Conv2d(32, 64, (3, 9), (1, 2), (1, 4)),
            nn.Conv2d(64, 128, (3, 9), (1, 2), (1, 4)),
            nn.Conv2d(128, 128, (3, 3), (1, 1), (1, 1)),
            nn.Conv2d(128, 1, (3, 3), (1, 1), (1, 1)),
        ])

    def forward(self, x: torch.Tensor):
        spec = torch.stft(x.float(), self.n_fft, self.hop, self.n_fft, self.win, return_complex=True).abs().unsqueeze(1)
        fmap = []
        h = spec
        for i, conv in enumerate(self.convs):
            h = conv(h)
            if i < len(self.convs) - 1:
                h = F.leaky_relu(h, 0.1)
            fmap.append(h)
        return h.flatten(1, -1), fmap


class LiteVocoderDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.discs = nn.ModuleList([
            PeriodDisc(2), PeriodDisc(3), PeriodDisc(5), PeriodDisc(7),
            SpecDisc(512, 128), SpecDisc(1024, 256), SpecDisc(2048, 512),
        ])

    def forward(self, y_real: torch.Tensor, y_fake: torch.Tensor):
        real_outs, fake_outs, real_fmaps, fake_fmaps = [], [], [], []
        for d in self.discs:
            ro, rf = d(y_real)
            fo, ff = d(y_fake)
            real_outs.append(ro); fake_outs.append(fo)
            real_fmaps.append(rf); fake_fmaps.append(ff)
        return real_outs, fake_outs, real_fmaps, fake_fmaps


def multi_res_stft_loss(y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
    loss = 0.0
    for n_fft, hop in [(512, 128), (1024, 256), (2048, 512)]:
        win = torch.hann_window(n_fft, device=y_pred.device)
        sp = torch.stft(y_pred.float(), n_fft, hop, n_fft, win, return_complex=True).abs().clamp(min=1e-5)
        st = torch.stft(y_true.float(), n_fft, hop, n_fft, win, return_complex=True).abs().clamp(min=1e-5)
        sc = torch.norm(st - sp, p="fro") / (torch.norm(st, p="fro") + 1e-6)
        mag = F.l1_loss(torch.log(sp), torch.log(st))
        loss = loss + sc + mag
    return loss / 3.0


def count_params_m(model: nn.Module) -> float:
    return sum(p.numel() for p in model.parameters()) / 1e6


## 6. 训练前硬门禁：参数量对齐检查 & 流式↔离线数值等价单元测试


In [ ]:
@torch.no_grad()
def run_architecture_and_streaming_tests():
    print("=" * 72)
    print("1) 参数量检查 & 流式↔离线等价性测试 (要求最大绝对误差 < 1e-4)")
    print("=" * 72)
    T = 23
    pids = torch.randint(4, FRONTEND.vocab_size, (1, T), dtype=torch.long)
    lids = torch.randint(0, 3, (1, T), dtype=torch.long)

    for exp_name, bb_type, lookahead in EXPERIMENTS:
        m = AetherAcousticModel(
            vocab_size=FRONTEND.vocab_size,
            backbone_type=bb_type,
            d_model=CFG["d_model"],
            n_mels=CFG["n_mels"],
            k_left=CFG["k_left"],
            lookahead=lookahead,
            swa_window=CFG["swa_window"],
        ).eval()
        # 推理期不计训练用 aligner 参数
        infer_params = sum(p.numel() for n, p in m.named_parameters() if not n.startswith("aligner.")) / 1e6
        mel_off, dur_off, _, _ = m.infer_offline(pids, lids)

        if m.is_streaming:
            st = m.init_stream_state(1, device=pids.device, dtype=torch.float32)
            chunks = []
            step = CFG["stream_text_chunk"]
            for i in range(0, T, step):
                is_last = (i + step >= T)
                mc, st = m.infer_stream_step(pids[:, i:i+step], lids[:, i:i+step], st, is_final=is_last)
                if mc.shape[-1] > 0:
                    chunks.append(mc)
            mel_str = torch.cat(chunks, dim=-1)
            assert mel_str.shape == mel_off.shape, f"{exp_name} shape mismatch: {mel_str.shape} vs {mel_off.shape}"
            diff = (mel_off - mel_str).abs().max().item()
            assert diff < 1e-4, f"{exp_name} streaming mismatch: {diff}"
            print(f"  [PASS] {exp_name:26s} | Infer Params: {infer_params:.2f}M | L={lookahead} | Max Stream Diff: {diff:.2e}")
        else:
            print(f"  [PASS] {exp_name:26s} | Infer Params: {infer_params:.2f}M | (Offline Upper-Bound)")

    # 测试 CausalVocosLite 流式 OLA 等价性
    voc = CausalVocosLite(n_mels=CFG["n_mels"], n_fft=CFG["n_fft"], hop_length=CFG["hop_length"]).eval()
    F_len = 53
    mel_dummy = torch.randn(1, CFG["n_mels"], F_len)
    wav_off = voc(mel_dummy)
    vst = voc.init_state(1, device=mel_dummy.device, dtype=torch.float32)
    w_chunks = []
    c_sz = CFG["stream_mel_chunk"]
    for i in range(0, F_len, c_sz):
        is_last = (i + c_sz >= F_len)
        wc, vst = voc.forward_step(mel_dummy[:, :, i:i+c_sz], vst, is_final=is_last)
        if wc.shape[-1] > 0:
            w_chunks.append(wc)
    wav_str = torch.cat(w_chunks, dim=-1)
    assert wav_str.shape == wav_off.shape, f"Vocoder shape mismatch: {wav_str.shape} vs {wav_off.shape}"
    vdiff = (wav_off - wav_str).abs().max().item()
    assert vdiff < 1e-4, f"Vocoder OLA mismatch: {vdiff}"
    print(f"  [PASS] {'CausalVocosLite (24kHz)':26s} | Infer Params: {count_params_m(voc):.2f}M | OLA Stream Diff: {vdiff:.2e}")
    print("=" * 72)


run_architecture_and_streaming_tests()


## 7. DataLoader 构建（自动加载规范化统计量）


In [ ]:
from torch.utils.data import Dataset, DataLoader

STATS = DATASET_META["stats"]
MEL_MEAN = torch.tensor(STATS["mel_mean"], dtype=torch.float32)[:, None]
MEL_STD = torch.tensor(STATS["mel_std"], dtype=torch.float32)[:, None]


def norm_mel(m): return (m - MEL_MEAN.to(m.device)) / MEL_STD.to(m.device)
def denorm_mel(m): return m * MEL_STD.to(m.device) + MEL_MEAN.to(m.device)


class AcousticDataset(Dataset):
    def __init__(self, entries):
        self.entries = entries

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        d = torch.load(self.entries[idx]["feat_path"], map_location="cpu", weights_only=False)
        mel = (d["mel"] - MEL_MEAN) / MEL_STD
        f0 = (d["log_f0"] - STATS["f0_mean"]) / STATS["f0_std"]
        energy = (d["log_energy"] - STATS["e_mean"]) / STATS["e_std"]
        return {
            "phoneme_ids": d["phoneme_ids"],
            "lang_ids": d["lang_ids"],
            "mel": mel,
            "f0": f0,
            "energy": energy,
        }


def acoustic_collate(batch):
    B = len(batch)
    T_max = max(b["phoneme_ids"].shape[0] for b in batch)
    F_max = max(b["mel"].shape[-1] for b in batch)
    pids = torch.zeros(B, T_max, dtype=torch.long)
    lids = torch.zeros(B, T_max, dtype=torch.long)
    t_lens = torch.zeros(B, dtype=torch.long)
    mels = torch.zeros(B, CFG["n_mels"], F_max, dtype=torch.float32)
    f0s = torch.zeros(B, F_max, dtype=torch.float32)
    es = torch.zeros(B, F_max, dtype=torch.float32)
    m_lens = torch.zeros(B, dtype=torch.long)
    for i, b in enumerate(batch):
        t = b["phoneme_ids"].shape[0]
        f = b["mel"].shape[-1]
        pids[i, :t] = b["phoneme_ids"]
        lids[i, :t] = b["lang_ids"]
        t_lens[i] = t
        mels[i, :, :f] = b["mel"]
        f0s[i, :f] = b["f0"]
        es[i, :f] = b["energy"]
        m_lens[i] = f
    return pids, lids, t_lens, mels, f0s, es, m_lens


class VocoderDataset(Dataset):
    def __init__(self, entries, seg_frames: int = 96):
        # 96 frames * 256 / 24000 ≈ 1.02s 片段，训练显存低且收敛快
        self.entries = entries
        self.seg_frames = seg_frames
        self.seg_samples = seg_frames * CFG["hop_length"]

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        d = torch.load(self.entries[idx]["feat_path"], map_location="cpu", weights_only=False)
        wav = d["wav"]
        if wav.shape[0] < self.seg_samples:
            wav = F.pad(wav, (0, self.seg_samples - wav.shape[0]))
            st = 0
        else:
            st = random.randint(0, wav.shape[0] - self.seg_samples)
        wav_seg = wav[st : st + self.seg_samples]
        mel_seg = MEL_EXTRACTOR(wav_seg)[:, :self.seg_frames]
        return mel_seg, wav_seg


train_loader = DataLoader(
    AcousticDataset(DATASET_META["train"]),
    batch_size=CFG["batch_size"],
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    collate_fn=acoustic_collate,
    drop_last=True,
)
val_loader = DataLoader(
    AcousticDataset(DATASET_META["val"]),
    batch_size=min(16, len(DATASET_META["val"])),
    shuffle=False,
    num_workers=2,
    collate_fn=acoustic_collate,
)
print(f"Train batches/epoch: {len(train_loader)} | Val batches: {len(val_loader)}")


## 8. 训练共享因果声码器 `CausalVocosLite`（并同步到 Google 云盘）& 加载预训练 Vocos 对照


In [ ]:
VOCODER_DRIVE_CKPT = DRIVE_ROOT / "vocoder" / f"causal_vocos_lite_{CFG['dataset_source']}_{CFG['profile']}.pt"
VOCODER_LOCAL_CKPT = LOCAL_WORK / "runs" / "causal_vocos_lite.pt"


def train_or_load_causal_vocoder():
    voc = CausalVocosLite(
        n_mels=CFG["n_mels"], n_fft=CFG["n_fft"], hop_length=CFG["hop_length"]
    ).to(DEVICE)

    if SKIP_IF_DONE_ON_DRIVE and VOCODER_DRIVE_CKPT.exists():
        print(f"[Drive Resume] 直接加载云盘已训因果声码器: {VOCODER_DRIVE_CKPT}")
        voc.load_state_dict(torch.load(VOCODER_DRIVE_CKPT, map_location=DEVICE, weights_only=True))
        return voc.eval()

    voc_loader = DataLoader(
        VocoderDataset(DATASET_META["train"], seg_frames=96),
        batch_size=min(16, CFG["batch_size"]),
        shuffle=True,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
        drop_last=True,
    )
    disc = LiteVocoderDiscriminator().to(DEVICE)
    opt_g = torch.optim.AdamW(voc.parameters(), lr=CFG["lr_vocoder"], betas=(0.8, 0.99))
    opt_d = torch.optim.AdamW(disc.parameters(), lr=CFG["lr_vocoder"], betas=(0.8, 0.99))
    mel_gpu = VocosCompatibleMel(CFG["sample_rate"], CFG["n_fft"], CFG["hop_length"], CFG["n_mels"]).to(DEVICE)

    s1_steps = CFG["vocoder_stage1_steps"]
    s2_steps = CFG["vocoder_stage2_steps"]
    total_steps = s1_steps + s2_steps
    step = 0
    t0 = time.time()
    pbar = tqdm(total=total_steps, desc="Train CausalVocosLite (Stage1 Spectral -> Stage2 GAN)")

    while step < total_steps:
        for mel_seg, wav_gt in voc_loader:
            if step >= total_steps:
                break
            mel_seg = mel_seg.to(DEVICE, non_blocking=True)
            wav_gt = wav_gt.to(DEVICE, non_blocking=True)

            in_gan = step >= s1_steps
            # 借鉴 Paradee 经验：进入对抗阶段后降低频谱权重，让判别器充分消除 2-8kHz 嗡鸣
            spec_weight = 5.0 if in_gan else 15.0

            wav_pred = voc(mel_seg)
            L_min = min(wav_pred.shape[-1], wav_gt.shape[-1])
            wav_pred, wav_gt = wav_pred[:, :L_min], wav_gt[:, :L_min]

            if in_gan:
                # 1) 更新判别器
                opt_d.zero_grad(set_to_none=True)
                ro, fo, _, _ = disc(wav_gt, wav_pred.detach())
                loss_d = sum(torch.mean((r - 1.0) ** 2) + torch.mean(f ** 2) for r, f in zip(ro, fo)) / len(ro)
                loss_d.backward()
                torch.nn.utils.clip_grad_norm_(disc.parameters(), 1.0)
                opt_d.step()

            # 2) 更新生成器
            opt_g.zero_grad(set_to_none=True)
            loss_stft = multi_res_stft_loss(wav_pred, wav_gt)
            loss_mel = F.l1_loss(mel_gpu(wav_pred), mel_gpu(wav_gt))
            loss_g = spec_weight * (loss_stft + loss_mel)
            if in_gan:
                _, fo, rfmaps, ffmaps = disc(wav_gt, wav_pred)
                loss_adv = sum(torch.mean((f - 1.0) ** 2) for f in fo) / len(fo)
                loss_fm = sum(
                    F.l1_loss(ff, rf.detach())
                    for rf_list, ff_list in zip(rfmaps, ffmaps)
                    for rf, ff in zip(rf_list, ff_list)
                ) / (len(rfmaps) * 5)
                loss_g = loss_g + loss_adv + 2.0 * loss_fm

            loss_g.backward()
            torch.nn.utils.clip_grad_norm_(voc.parameters(), 1.0)
            opt_g.step()

            step += 1
            pbar.update(1)
            if step % 500 == 0 or step == total_steps:
                pbar.set_postfix(stage="GAN" if in_gan else "SPEC", stft=f"{loss_stft.item():.3f}", mel=f"{loss_mel.item():.3f}")

    pbar.close()
    torch.save(voc.state_dict(), VOCODER_LOCAL_CKPT)
    sync_file_to_drive(VOCODER_LOCAL_CKPT, VOCODER_DRIVE_CKPT)
    print(f"[Saved] CausalVocosLite 训练完成（耗时 {(time.time()-t0)/60:.1f} min），已同步到云盘: {VOCODER_DRIVE_CKPT}")
    return voc.eval()


CAUSAL_VOCODER = None
PRETRAINED_VOCOS = None

if CFG["vocoder_mode"] in ("both", "causal_vocos_lite"):
    CAUSAL_VOCODER = train_or_load_causal_vocoder()

if CFG["vocoder_mode"] in ("both", "pretrained_vocos"):
    try:
        from vocos import Vocos
        print("[Load] 加载预训练 charactr/vocos-mel-24khz 作为高质量声码器锚点...")
        PRETRAINED_VOCOS = Vocos.from_pretrained("charactr/vocos-mel-24khz").to(DEVICE).eval()
    except Exception as e:
        print(f"[Warn] 预训练 Vocos 加载失败 ({e})，将仅使用自研 CausalVocosLite。")


## 9. 批量训练所有声学主干（`E0`–`E4`）并自动备份权重到 Google 云盘


In [ ]:
TRAIN_HISTORIES = {}


def train_one_model(exp_name: str, backbone_type: str, lookahead: int):
    run_local = LOCAL_WORK / "runs" / exp_name
    run_drive = DRIVE_ROOT / "runs" / f"{exp_name}_{CFG['dataset_source']}_{CFG['profile']}"
    run_local.mkdir(parents=True, exist_ok=True)
    run_drive.mkdir(parents=True, exist_ok=True)

    drive_best = run_drive / "best.pt"
    drive_hist = run_drive / "history.json"
    local_best = run_local / "best.pt"

    model = AetherAcousticModel(
        vocab_size=FRONTEND.vocab_size,
        backbone_type=backbone_type,
        d_model=CFG["d_model"],
        n_mels=CFG["n_mels"],
        k_left=CFG["k_left"],
        lookahead=lookahead,
        swa_window=CFG["swa_window"],
    ).to(DEVICE)

    if SKIP_IF_DONE_ON_DRIVE and drive_best.exists():
        print(f"\n[Drive Resume] 跳过已训练模型 {exp_name}，从云盘加载: {drive_best}")
        ckpt = torch.load(drive_best, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt["model"])
        hist = json.loads(drive_hist.read_text()) if drive_hist.exists() else {"steps": [], "val_mel": []}
        TRAIN_HISTORIES[exp_name] = hist
        return model.eval()

    print(f"\n{'='*72}")
    print(f"开始训练: {exp_name} (backbone={backbone_type}, lookahead={lookahead}, params={count_params_m(model):.2f}M)")
    print(f"{'='*72}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr_acoustic"], betas=(0.9, 0.98), weight_decay=1e-2)
    total_steps = CFG["acoustic_steps"]
    warmup = max(200, int(total_steps * 0.05))

    def lr_lambda(s):
        if s < warmup:
            return float(s + 1) / float(warmup)
        prog = float(s - warmup) / float(max(1, total_steps - warmup))
        return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * prog))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda" and AMP_DTYPE == torch.float16))

    history = {"steps": [], "train_loss": [], "val_mel": [], "val_dur": [], "elapsed_min": 0.0}
    best_val_mel = float("inf")
    step = 0
    t0 = time.time()
    eval_every = max(300, total_steps // 10)
    pbar = tqdm(total=total_steps, desc=f"Train {exp_name}")

    while step < total_steps:
        model.train()
        for pids, lids, t_lens, mels, f0s, es, m_lens in train_loader:
            if step >= total_steps:
                break
            pids = pids.to(DEVICE, non_blocking=True)
            lids = lids.to(DEVICE, non_blocking=True)
            t_lens = t_lens.to(DEVICE, non_blocking=True)
            mels = mels.to(DEVICE, non_blocking=True)
            f0s = f0s.to(DEVICE, non_blocking=True)
            es = es.to(DEVICE, non_blocking=True)
            m_lens = m_lens.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", dtype=AMP_DTYPE, enabled=(DEVICE.type == "cuda")):
                out = model.forward_train(pids, lids, t_lens, mels, f0s, es, m_lens)
                loss = out["loss"]

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            step += 1
            pbar.update(1)
            if step % 100 == 0:
                pbar.set_postfix(
                    loss=f"{loss.item():.3f}",
                    mel=f"{out['mel'].item():.3f}",
                    dur=f"{out['dur'].item():.3f}",
                    aln=f"{out['align'].item():.3f}",
                )

            if step % eval_every == 0 or step == total_steps:
                model.eval()
                v_mel, v_dur, v_cnt = 0.0, 0.0, 0
                with torch.no_grad():
                    for vp, vl, vtl, vm, vf0, ve, vml in val_loader:
                        vp, vl, vtl = vp.to(DEVICE), vl.to(DEVICE), vtl.to(DEVICE)
                        vm, vf0, ve, vml = vm.to(DEVICE), vf0.to(DEVICE), ve.to(DEVICE), vml.to(DEVICE)
                        vo = model.forward_train(vp, vl, vtl, vm, vf0, ve, vml)
                        v_mel += vo["mel"].item()
                        v_dur += vo["dur"].item()
                        v_cnt += 1
                v_mel /= max(v_cnt, 1)
                v_dur /= max(v_cnt, 1)
                history["steps"].append(step)
                history["train_loss"].append(round(loss.item(), 4))
                history["val_mel"].append(round(v_mel, 4))
                history["val_dur"].append(round(v_dur, 4))
                history["elapsed_min"] = round((time.time() - t0) / 60.0, 2)

                if v_mel < best_val_mel:
                    best_val_mel = v_mel
                    payload = {
                        "exp_name": exp_name,
                        "backbone_type": backbone_type,
                        "lookahead": lookahead,
                        "step": step,
                        "best_val_mel": best_val_mel,
                        "cfg": CFG,
                        "stats": STATS,
                        "vocab": FRONTEND.vocab,
                        "model": model.state_dict(),
                    }
                    torch.save(payload, local_best)
                    sync_file_to_drive(local_best, drive_best)
                model.train()

    pbar.close()
    # 保存训练曲线与动态 INT8 量化包到 Google Drive
    (run_local / "history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
    sync_file_to_drive(run_local / "history.json", drive_hist)

    ckpt = torch.load(local_best, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["model"])

    # 生成动态 INT8 量化参考包并同步到 Drive（评估端侧体积）
    model_cpu = model.to("cpu").eval()
    try:
        q_model = torch.ao.quantization.quantize_dynamic(model_cpu, {nn.Linear, nn.GRU}, dtype=torch.qint8)
        local_int8 = run_local / "model_int8.pt"
        torch.save(q_model.state_dict(), local_int8)
        sync_file_to_drive(local_int8, run_drive / "model_int8.pt")
    except Exception as e:
        print(f"[Warn] INT8 dynamic export skipped for {exp_name}: {e}")

    model = model.to(DEVICE).eval()
    TRAIN_HISTORIES[exp_name] = history
    print(f"[Done] {exp_name} | Best Val Mel L1: {best_val_mel:.4f} | Time: {history['elapsed_min']} min | 已同步云盘: {run_drive}")
    return model


TRAINED_MODELS = {}
for exp_name, bb_type, la in EXPERIMENTS:
    TRAINED_MODELS[exp_name] = train_one_model(exp_name, bb_type, la)


## 10. 批量生成测试音频（含真流式逐块合成）并拷贝到 Google 云盘


In [ ]:
# 50 句黄金测试集（中/英/混读/多音字/数字/短句/长句），自动匹配当前训练语料语种
GOLDEN_ZH = [
    ("zh_qa_01", "你好，请问今天北京的天气怎么样？"),
    ("zh_qa_02", "好的，已经帮你把明早七点的闹钟设置好了。"),
    ("zh_poly_01", "中国人民银行行长今天上午发表了重要讲话。"),
    ("zh_poly_02", "这件事情非常重，我们不要重复同样的错误。"),
    ("zh_sandhi_01", "你好，我想去展览馆买把好雨伞。"),
    ("zh_tn_01", "今年是2026年，同比增长了18.5%。"),
    ("zh_short_01", "好的，没问题。"),
    ("zh_short_02", "收到，马上处理。"),
    ("zh_long_01", "在嵌入式设备上实现端侧语音合成时，需要在自然度、首包延迟和纯中央处理器算力占用之间找到一个非常精准的平衡点。"),
    ("zh_cs_01", "帮我把 Wi-Fi 切换到 5G，顺便查一下今天的 API latency。"),
]
GOLDEN_EN = [
    ("en_qa_01", "Hello, how is the weather in San Francisco today?"),
    ("en_qa_02", "Sure, I have set your alarm for seven thirty tomorrow morning."),
    ("en_short_01", "Got it, no problem."),
    ("en_short_02", "Sure thing."),
    ("en_tech_01", "Streaming text to speech requires low latency and consistent speaker identity."),
    ("en_long_01", "Running neural speech synthesis directly on an embedded Linux processor eliminates cloud round trips while protecting user privacy."),
]


def build_eval_suite():
    suite = []
    # 1) 优先从 hold-out val 集抽样（带真实参考 WAV，可算 SIM、高频对比和真声 Gold Anchor！）
    val_entries = DATASET_META["val"][: CFG["eval_max_utts"]]
    for e in val_entries:
        suite.append({
            "utt_id": f"val_{e['utt_id']}",
            "text": e["text"],
            "feat_path": e["feat_path"],
            "is_val": True,
        })
    # 2) 追加黄金压力测试句（无 Ground Truth WAV，专测多音字/短句/长句/混读）
    if CFG["dataset_source"] == "bznsyp":
        extras = GOLDEN_ZH
    elif CFG["dataset_source"] == "ljspeech":
        extras = GOLDEN_EN
    else:
        extras = GOLDEN_ZH + GOLDEN_EN
    for uid, txt in extras:
        suite.append({"utt_id": uid, "text": txt, "feat_path": None, "is_val": False})
    return suite


EVAL_SUITE = build_eval_suite()
print(f"评测集构建完成：共 {len(EVAL_SUITE)} 句（其中带真声参考的 Val 句: {sum(x['is_val'] for x in EVAL_SUITE)} 句）")


@torch.no_grad()
def synthesize_offline(model: AetherAcousticModel, text: str, vocoder, use_pretrained_vocos: bool = False):
    pids, lids = FRONTEND.text_to_ids(text)
    pt = torch.tensor([pids], dtype=torch.long, device=DEVICE)
    lt = torch.tensor([lids], dtype=torch.long, device=DEVICE)
    mel_norm, dur, f0, energy = model.infer_offline(pt, lt)
    mel = denorm_mel(mel_norm)
    if use_pretrained_vocos and PRETRAINED_VOCOS is not None:
        wav = PRETRAINED_VOCOS.decode(mel)
    else:
        wav = vocoder(mel)
    return wav.squeeze(0).cpu().float().numpy(), mel.squeeze(0).cpu().float().numpy()


@torch.no_grad()
def synthesize_streaming(model: AetherAcousticModel, text: str, vocoder: CausalVocosLite):
    """模拟 LLM 流式输入 -> 流式音素块 -> 流式 Mel 块 -> CausalVocosLite 零爆音 OLA 流式输出，记录切块边界位置"""
    pids, lids = FRONTEND.text_to_ids(text)
    pt = torch.tensor([pids], dtype=torch.long, device=DEVICE)
    lt = torch.tensor([lids], dtype=torch.long, device=DEVICE)
    T = pt.shape[1]

    ast_state = model.init_stream_state(1, device=DEVICE, dtype=torch.float32)
    voc_state = vocoder.init_state(1, device=DEVICE, dtype=torch.float32)

    mel_buf = torch.zeros(1, CFG["n_mels"], 0, device=DEVICE, dtype=torch.float32)
    wav_chunks = []
    mel_chunks = []
    seam_positions = []
    cur_samples = 0

    t_step = CFG["stream_text_chunk"]
    m_step = CFG["stream_mel_chunk"]

    for i in range(0, T, t_step):
        is_last = (i + t_step >= T)
        mc, ast_state = model.infer_stream_step(pt[:, i:i+t_step], lt[:, i:i+t_step], ast_state, is_final=is_last)
        if mc.shape[-1] > 0:
            mc_denorm = denorm_mel(mc)
            mel_chunks.append(mc_denorm)
            mel_buf = torch.cat([mel_buf, mc_denorm], dim=-1)

        while mel_buf.shape[-1] >= m_step:
            chunk_in = mel_buf[:, :, :m_step]
            mel_buf = mel_buf[:, :, m_step:]
            wc, voc_state = vocoder.forward_step(chunk_in, voc_state, is_final=False)
            if wc.shape[-1] > 0:
                if cur_samples > 0:
                    seam_positions.append(cur_samples)
                wav_chunks.append(wc.squeeze(0).cpu().float().numpy())
                cur_samples += wc.shape[-1]

        if is_last:
            wc, voc_state = vocoder.forward_step(mel_buf, voc_state, is_final=True)
            if wc.shape[-1] > 0:
                if cur_samples > 0:
                    seam_positions.append(cur_samples)
                wav_chunks.append(wc.squeeze(0).cpu().float().numpy())
                cur_samples += wc.shape[-1]

    wav = np.concatenate(wav_chunks) if wav_chunks else np.zeros(2400, dtype=np.float32)
    mel_all = torch.cat(mel_chunks, dim=-1).squeeze(0).cpu().float().numpy() if mel_chunks else np.zeros((CFG["n_mels"], 1))
    return wav, mel_all, seam_positions


# 生成全部测试音频并同步到 Google Drive
EVAL_LOCAL_DIR = LOCAL_WORK / "eval_samples" / f"{CFG['dataset_source']}_{CFG['profile']}"
EVAL_DRIVE_DIR = DRIVE_ROOT / "eval_samples" / f"{CFG['dataset_source']}_{CFG['profile']}"
EVAL_LOCAL_DIR.mkdir(parents=True, exist_ok=True)

# 1) 导出 Ground-Truth 参考音频（Gold Anchor）
gt_dir = EVAL_LOCAL_DIR / "00_ground_truth"
gt_dir.mkdir(parents=True, exist_ok=True)
for item in EVAL_SUITE:
    if item["feat_path"] is not None:
        d = torch.load(item["feat_path"], map_location="cpu", weights_only=False)
        sf.write(gt_dir / f"{item['utt_id']}.wav", d["wav"].numpy(), CFG["sample_rate"])

# 2) 批量生成每个模型的离线音频 + 真流式音频
GENERATED_RECORDS = {}  # exp_name -> list of dicts
primary_vocoder = CAUSAL_VOCODER
use_pre_for_offline = (CAUSAL_VOCODER is None)

for exp_name, model in TRAINED_MODELS.items():
    exp_out_off = EVAL_LOCAL_DIR / exp_name / "offline"
    exp_out_str = EVAL_LOCAL_DIR / exp_name / "streaming"
    exp_out_pre = EVAL_LOCAL_DIR / exp_name / "pretrained_vocos"
    exp_out_off.mkdir(parents=True, exist_ok=True)
    if model.is_streaming and CAUSAL_VOCODER is not None:
        exp_out_str.mkdir(parents=True, exist_ok=True)
    if PRETRAINED_VOCOS is not None:
        exp_out_pre.mkdir(parents=True, exist_ok=True)

    recs = []
    for item in tqdm(EVAL_SUITE, desc=f"Synthesize {exp_name}"):
        wav_off, mel_off = synthesize_offline(
            model, item["text"],
            vocoder=primary_vocoder if not use_pre_for_offline else PRETRAINED_VOCOS,
            use_pretrained_vocos=use_pre_for_offline,
        )
        p_off = exp_out_off / f"{item['utt_id']}.wav"
        sf.write(p_off, np.clip(wav_off, -1.0, 1.0), CFG["sample_rate"])

        p_pre = None
        if PRETRAINED_VOCOS is not None and not use_pre_for_offline:
            wav_pre, _ = synthesize_offline(model, item["text"], vocoder=None, use_pretrained_vocos=True)
            p_pre = exp_out_pre / f"{item['utt_id']}.wav"
            sf.write(p_pre, np.clip(wav_pre, -1.0, 1.0), CFG["sample_rate"])

        p_str, seams, stream_diff = None, [], 0.0
        if model.is_streaming and CAUSAL_VOCODER is not None:
            wav_str, mel_str, seams = synthesize_streaming(model, item["text"], CAUSAL_VOCODER)
            p_str = exp_out_str / f"{item['utt_id']}.wav"
            sf.write(p_str, np.clip(wav_str, -1.0, 1.0), CFG["sample_rate"])
            L_min = min(len(wav_off), len(wav_str))
            stream_diff = float(np.max(np.abs(wav_off[:L_min] - wav_str[:L_min]))) if L_min > 0 else 0.0

        recs.append({
            "utt_id": item["utt_id"],
            "text": item["text"],
            "is_val": item["is_val"],
            "gt_wav": str(gt_dir / f"{item['utt_id']}.wav") if item["is_val"] else None,
            "off_wav": str(p_off),
            "str_wav": str(p_str) if p_str else None,
            "pre_wav": str(p_pre) if p_pre else None,
            "seams": seams,
            "stream_wav_max_diff": stream_diff,
        })
    GENERATED_RECORDS[exp_name] = recs

print("[Sync] 正在将所有生成音频拷贝到 Google 云盘...")
sync_dir_to_drive(EVAL_LOCAL_DIR, EVAL_DRIVE_DIR)
print(f"[Synced] 全部生成音频已保存至 Google 云盘: {EVAL_DRIVE_DIR}")


## 11. 多维自动化质量打分 & 嵌入式纯 CPU（2 线程）RTF / 首包延迟基准


In [ ]:
import pandas as pd
from jiwer import cer as calc_cer, wer as calc_wer

# ------------------------------------------------------------------------------
# 11.1 加载自动评测器（容错封装，单个工具网络失败不中断总评分）
# ------------------------------------------------------------------------------
UTMOS_MODEL = None
try:
    print("[Eval] 加载 UTMOS (utmos22_strong)...")
    UTMOS_MODEL = torch.hub.load("tarepan/SpeechMOS:v1.2.0", "utmos22_strong", trust_repo=True).to(DEVICE).eval()
except Exception as e:
    print(f"[Warn] UTMOS unavailable: {e}")

SQUIM_MODEL = None
try:
    print("[Eval] 加载 TorchAudio SQUIM_OBJECTIVE (PESQ / SI-SDR / STOI)...")
    SQUIM_MODEL = torchaudio.pipelines.SQUIM_OBJECTIVE.get_model().to(DEVICE).eval()
except Exception as e:
    print(f"[Warn] SQUIM unavailable: {e}")

SPK_ENCODER = None
try:
    from resemblyzer import VoiceEncoder, preprocess_wav
    print("[Eval] 加载 Resemblyzer 声纹编码器 (Speaker SIM)...")
    SPK_ENCODER = VoiceEncoder(device=str(DEVICE))
except Exception as e:
    print(f"[Warn] Resemblyzer unavailable: {e}")

WHISPER_MODEL = None
try:
    from faster_whisper import WhisperModel
    w_name = "small" if CFG["profile"] == "quick_smoke" else "large-v3-turbo"
    print(f"[Eval] 加载 Faster-Whisper ({w_name}) 评测 CER/WER/MER...")
    WHISPER_MODEL = WhisperModel(w_name, device="cuda" if torch.cuda.is_available() else "cpu", compute_type="float16" if torch.cuda.is_available() else "int8")
except Exception as e:
    print(f"[Warn] Faster-Whisper unavailable: {e}")


def normalize_for_asr(text: str) -> str:
    text = unicodedata.normalize("NFKC", text).lower()
    text = re.sub(r"[^\u4e00-\u9fffa-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def compute_mixed_error_rate(ref: str, hyp: str) -> float:
    """中文按字、英文按词计算混合错误率 MER"""
    def tokenize(s):
        s = normalize_for_asr(s)
        return re.findall(r"[\u4e00-\u9fff]|[a-z0-9]+", s)
    r_toks = tokenize(ref)
    h_toks = tokenize(hyp)
    if not r_toks:
        return 0.0
    return float(calc_wer(" ".join(r_toks), " ".join(h_toks)))


@torch.no_grad()
def score_utmos(wav_np: np.ndarray, sr: int = 24000) -> float:
    if UTMOS_MODEL is None:
        return float("nan")
    w16 = librosa.resample(wav_np, orig_sr=sr, target_sr=16000)
    wt = torch.from_numpy(w16).unsqueeze(0).to(DEVICE)
    return float(UTMOS_MODEL(wt, 16000).item())


@torch.no_grad()
def score_squim(wav_np: np.ndarray, sr: int = 24000):
    if SQUIM_MODEL is None:
        return float("nan"), float("nan"), float("nan")
    w16 = librosa.resample(wav_np, orig_sr=sr, target_sr=16000)
    wt = torch.from_numpy(w16).unsqueeze(0).to(DEVICE)
    stoi, pesq, sisdr = SQUIM_MODEL(wt)
    return float(pesq.item()), float(sisdr.item()), float(stoi.item())


def hf_2_8k_ratio(wav_np: np.ndarray, sr: int = 24000) -> float:
    """计算 2-8 kHz 频段能量占比，用于监控瘦身声码器高频发闷或嗡嗡声异常"""
    spec = np.abs(np.fft.rfft(wav_np)) ** 2
    freqs = np.fft.rfftfreq(len(wav_np), 1.0 / sr)
    band = (freqs >= 2000.0) & (freqs <= 8000.0)
    return float(spec[band].sum() / (spec.sum() + 1e-8))


def detect_seam_click(wav_np: np.ndarray, seams: list, sr: int = 24000) -> float:
    """检测每个 chunk 拼接点 ±5ms 内是否出现一阶差分尖峰（爆音检测）"""
    if not seams:
        return 0.0
    diff = np.abs(np.diff(wav_np))
    win = int(0.005 * sr)
    global_rms = float(np.sqrt(np.mean(diff ** 2) + 1e-10))
    clicks = 0
    for pos in seams:
        lo = max(0, pos - win)
        hi = min(len(diff), pos + win)
        if hi <= lo:
            continue
        local_max = float(np.max(diff[lo:hi]))
        # 同时对比两侧邻域，过滤正常清辅音爆破
        ref_lo = max(0, pos - 4 * win)
        ref_hi = min(len(diff), pos + 4 * win)
        local_rms = float(np.sqrt(np.mean(diff[ref_lo:ref_hi] ** 2) + 1e-10))
        if local_max > max(12.0 * local_rms, 15.0 * global_rms):
            clicks += 1
    return float(clicks) / len(seams)


def compute_asr_error(wav_path: str, ref_text: str) -> float:
    if WHISPER_MODEL is None:
        return float("nan")
    lang = "zh" if re.search(r"[\u4e00-\u9fff]", ref_text) else "en"
    segs, _ = WHISPER_MODEL.transcribe(wav_path, language=lang, beam_size=1)
    hyp = "".join(s.text for s in segs)
    return compute_mixed_error_rate(ref_text, hyp)


# 计算 Ground-Truth 参考声纹均值 & 基准高频比
GT_REF_EMBS = []
GT_HF_RATIOS = []
GT_UTMOS_LIST = []
for item in EVAL_SUITE:
    if item["is_val"]:
        p = EVAL_LOCAL_DIR / "00_ground_truth" / f"{item['utt_id']}.wav"
        w, _ = sf.read(str(p), dtype="float32")
        GT_HF_RATIOS.append(hf_2_8k_ratio(w))
        GT_UTMOS_LIST.append(score_utmos(w))
        if SPK_ENCODER is not None:
            GT_REF_EMBS.append(SPK_ENCODER.embed_utterance(preprocess_wav(w, source_sr=CFG["sample_rate"])))

GT_SPK_CENTROID = np.mean(GT_REF_EMBS, axis=0) if GT_REF_EMBS else None
if GT_SPK_CENTROID is not None:
    GT_SPK_CENTROID /= (np.linalg.norm(GT_SPK_CENTROID) + 1e-8)
GT_HF_MEAN = float(np.mean(GT_HF_RATIOS)) if GT_HF_RATIOS else 0.15
GT_UTMOS_MEAN = float(np.nanmean(GT_UTMOS_LIST)) if GT_UTMOS_LIST else float("nan")
print(f"[Gold Anchor] Ground-Truth UTMOS: {GT_UTMOS_MEAN:.3f} | 2-8kHz Energy Ratio: {GT_HF_MEAN:.4f}")


# ------------------------------------------------------------------------------
# 11.2 纯 CPU 2 线程 RTF & 首包音频延迟 TTFA 基准测试
# ------------------------------------------------------------------------------
@torch.no_grad()
def benchmark_cpu_2thread(model: AetherAcousticModel, vocoder: CausalVocosLite, sample_text: str):
    prev_threads = torch.get_num_threads()
    torch.set_num_threads(2)
    m_cpu = model.to("cpu").eval()
    v_cpu = vocoder.to("cpu").eval() if vocoder is not None else None

    pids, lids = FRONTEND.text_to_ids(sample_text)
    pt = torch.tensor([pids], dtype=torch.long)
    lt = torch.tensor([lids], dtype=torch.long)
    T = pt.shape[1]

    # Warmup
    _ = m_cpu.infer_offline(pt, lt)

    if m_cpu.is_streaming and v_cpu is not None:
        t_start = time.perf_counter()
        ttfa_ms = None
        ast_st = m_cpu.init_stream_state(1, device=torch.device("cpu"), dtype=torch.float32)
        voc_st = v_cpu.init_state(1, device=torch.device("cpu"), dtype=torch.float32)
        mel_buf = torch.zeros(1, CFG["n_mels"], 0)
        total_samples = 0
        t_step = CFG["stream_text_chunk"]
        m_step = CFG["stream_mel_chunk"]
        for i in range(0, T, t_step):
            is_last = (i + t_step >= T)
            mc, ast_st = m_cpu.infer_stream_step(pt[:, i:i+t_step], lt[:, i:i+t_step], ast_st, is_final=is_last)
            if mc.shape[-1] > 0:
                mel_buf = torch.cat([mel_buf, denorm_mel(mc)], dim=-1)
            while mel_buf.shape[-1] >= m_step:
                wc, voc_st = v_cpu.forward_step(mel_buf[:, :, :m_step], voc_st, is_final=False)
                mel_buf = mel_buf[:, :, m_step:]
                total_samples += wc.shape[-1]
                if ttfa_ms is None and wc.shape[-1] > 0:
                    ttfa_ms = (time.perf_counter() - t_start) * 1000.0
            if is_last:
                wc, voc_st = v_cpu.forward_step(mel_buf, voc_st, is_final=True)
                total_samples += wc.shape[-1]
                if ttfa_ms is None and wc.shape[-1] > 0:
                    ttfa_ms = (time.perf_counter() - t_start) * 1000.0
        elapsed = time.perf_counter() - t_start
        audio_sec = max(total_samples / CFG["sample_rate"], 1e-3)
        rtf = elapsed / audio_sec
    else:
        t_start = time.perf_counter()
        mel_n, _, _, _ = m_cpu.infer_offline(pt, lt)
        mel = denorm_mel(mel_n)
        if v_cpu is not None:
            wav = v_cpu(mel)
            audio_sec = max(wav.shape[-1] / CFG["sample_rate"], 1e-3)
        else:
            audio_sec = max(mel.shape[-1] * CFG["hop_length"] / CFG["sample_rate"], 1e-3)
        elapsed = time.perf_counter() - t_start
        rtf = elapsed / audio_sec
        ttfa_ms = elapsed * 1000.0  # 非流式首包=整句耗时

    model.to(DEVICE)
    if vocoder is not None:
        vocoder.to(DEVICE)
    torch.set_num_threads(prev_threads)
    return round(rtf, 4), round(ttfa_ms or 0.0, 1)


# ------------------------------------------------------------------------------
# 11.3 汇总全部模型四维打分 + 综合排名
# ------------------------------------------------------------------------------
all_sample_rows = []
summary_rows = []
bench_text = GOLDEN_ZH[-2][1] if CFG["dataset_source"] != "ljspeech" else GOLDEN_EN[-1][1]

for exp_name, model in TRAINED_MODELS.items():
    recs = GENERATED_RECORDS[exp_name]
    utmos_s, pesq_s, sisdr_s, stoi_s, mer_s, sim_s, hf_diff_s, click_s, sdiff_s = [], [], [], [], [], [], [], [], []
    utmos_pre_s = []

    for r in tqdm(recs, desc=f"Scoring {exp_name}"):
        # 优先评测真流式 WAV（若为离线模型则测离线 WAV）
        eval_wav_path = r["str_wav"] if r["str_wav"] else r["off_wav"]
        wav, _ = sf.read(eval_wav_path, dtype="float32")

        u = score_utmos(wav)
        pesq, sisdr, stoi = score_squim(wav)
        mer = compute_asr_error(eval_wav_path, r["text"])
        hfd = abs(hf_2_8k_ratio(wav) - GT_HF_MEAN)
        clk = detect_seam_click(wav, r["seams"]) if r["str_wav"] else 0.0

        sim = float("nan")
        if SPK_ENCODER is not None and GT_SPK_CENTROID is not None:
            emb = SPK_ENCODER.embed_utterance(preprocess_wav(wav, source_sr=CFG["sample_rate"]))
            emb /= (np.linalg.norm(emb) + 1e-8)
            sim = float(np.dot(emb, GT_SPK_CENTROID))

        u_pre = float("nan")
        if r["pre_wav"]:
            w_pre, _ = sf.read(r["pre_wav"], dtype="float32")
            u_pre = score_utmos(w_pre)
            utmos_pre_s.append(u_pre)

        utmos_s.append(u); pesq_s.append(pesq); sisdr_s.append(sisdr); stoi_s.append(stoi)
        mer_s.append(mer); sim_s.append(sim); hf_diff_s.append(hfd); click_s.append(clk)
        sdiff_s.append(r["stream_wav_max_diff"])

        all_sample_rows.append({
            "model": exp_name, "utt_id": r["utt_id"], "text": r["text"],
            "UTMOS": u, "UTMOS_PreVocos": u_pre, "PESQ": pesq, "SI_SDR": sisdr,
            "MER": mer, "SIM": sim, "HF_2_8k_Diff": hfd, "Seam_Click": clk,
        })

    rtf_2t, ttfa_ms = benchmark_cpu_2thread(model, CAUSAL_VOCODER, bench_text)
    infer_params_m = sum(p.numel() for n, p in model.named_parameters() if not n.startswith("aligner.")) / 1e6
    int8_path = DRIVE_ROOT / "runs" / f"{exp_name}_{CFG['dataset_source']}_{CFG['profile']}" / "model_int8.pt"
    size_mb = round(int8_path.stat().st_size / 1e6, 2) if int8_path.exists() else round(infer_params_m * 4.0, 2)

    mean_utmos = float(np.nanmean(utmos_s))
    mean_utmos_pre = float(np.nanmean(utmos_pre_s)) if utmos_pre_s else float("nan")
    mean_mer = float(np.nanmean(mer_s))
    mean_sim = float(np.nanmean(sim_s))
    mean_pesq = float(np.nanmean(pesq_s))
    mean_click = float(np.mean(click_s))

    # 综合加权总分 (0-100)：自然度 35% + 可懂度 30% + 音色一致 15% + 速度/首包 20% - 爆音惩罚
    s_nat = np.clip((mean_utmos - 1.5) / 3.0, 0.0, 1.0) * 35.0 if not np.isnan(mean_utmos) else 20.0
    s_int = np.clip(1.0 - mean_mer * 3.0, 0.0, 1.0) * 30.0 if not np.isnan(mean_mer) else 20.0
    s_sim = np.clip((mean_sim - 0.5) / 0.45, 0.0, 1.0) * 15.0 if not np.isnan(mean_sim) else 10.0
    s_lat = np.clip(1.0 - ttfa_ms / 500.0, 0.0, 1.0) * 10.0 + np.clip(1.0 - rtf_2t / 0.5, 0.0, 1.0) * 10.0
    composite = round(float(s_nat + s_int + s_sim + s_lat - 20.0 * mean_click), 2)

    summary_rows.append({
        "Model": exp_name,
        "Streaming": model.is_streaming,
        "Lookahead": model.lookahead,
        "Params_M": round(infer_params_m, 2),
        "Size_MB": size_mb,
        "UTMOS↑": round(mean_utmos, 3),
        "UTMOS_PreVocos↑": round(mean_utmos_pre, 3) if not np.isnan(mean_utmos_pre) else None,
        "MER/CER↓": round(mean_mer, 4),
        "SIM↑": round(mean_sim, 3),
        "SQUIM_PESQ↑": round(mean_pesq, 3),
        "HF_2_8k_Diff↓": round(float(np.mean(hf_diff_s)), 4),
        "Seam_Click↓": round(mean_click, 4),
        "Stream_MaxDiff": f"{max(sdiff_s):.1e}",
        "CPU2T_RTF↓": rtf_2t,
        "CPU2T_TTFA_ms↓": ttfa_ms,
        "Composite_Score↑": composite,
    })

LEADERBOARD = pd.DataFrame(summary_rows).sort_values("Composite_Score↑", ascending=False).reset_index(drop=True)
PER_SAMPLE_DF = pd.DataFrame(all_sample_rows)

report_local = LOCAL_WORK / "reports"
report_drive = DRIVE_ROOT / "reports" / f"{CFG['dataset_source']}_{CFG['profile']}"
report_local.mkdir(parents=True, exist_ok=True)
report_drive.mkdir(parents=True, exist_ok=True)

LEADERBOARD.to_csv(report_local / "leaderboard.csv", index=False)
LEADERBOARD.to_json(report_local / "leaderboard.json", orient="records", force_ascii=False, indent=2)
PER_SAMPLE_DF.to_csv(report_local / "per_sample_scores.csv", index=False)
sync_dir_to_drive(report_local, report_drive)

print("\n================ AetherTTS Phase 1 综合评测排行榜 ================")
display(LEADERBOARD)
print(f"\n[Saved] 评测报告已同步到 Google 云盘: {report_drive}")


## 12. 可视化对比图 & 盲听试听面板（同步图表和盲听表到 Google 云盘）


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Audio, display, HTML

# 1) 绘制训练收敛曲线 & 多指标对比图
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 子图 1：验证集 Mel L1 收敛曲线
ax = axes[0]
for exp_name, hist in TRAIN_HISTORIES.items():
    if hist.get("steps"):
        ax.plot(hist["steps"], hist["val_mel"], marker="o", label=exp_name)
ax.set_title("Validation Mel L1 Convergence (Lower is Better)")
ax.set_xlabel("Steps"); ax.set_ylabel("Val Mel L1"); ax.grid(True, alpha=0.3); ax.legend(fontsize=8)

# 子图 2：自然度 UTMOS vs 综合分
ax = axes[1]
x = np.arange(len(LEADERBOARD))
ax.bar(x - 0.18, LEADERBOARD["UTMOS↑"].fillna(0), width=0.36, label="UTMOS (CausalVocosLite)")
if LEADERBOARD["UTMOS_PreVocos↑"].notna().any():
    ax.bar(x + 0.18, LEADERBOARD["UTMOS_PreVocos↑"].fillna(0), width=0.36, label="UTMOS (Pretrained Vocos)")
if not np.isnan(GT_UTMOS_MEAN):
    ax.axhline(GT_UTMOS_MEAN, color="red", linestyle="--", label=f"Gold GT ({GT_UTMOS_MEAN:.2f})")
ax.set_xticks(x); ax.set_xticklabels(LEADERBOARD["Model"], rotation=25, ha="right", fontsize=8)
ax.set_title("Naturalness (UTMOS)"); ax.set_ylim(0, 5.0); ax.grid(True, axis="y", alpha=0.3); ax.legend(fontsize=8)

# 子图 3：纯 CPU 2 线程 首包延迟 TTFA (ms) & RTF
ax = axes[2]
ax.bar(x, LEADERBOARD["CPU2T_TTFA_ms↓"], color="#4C78A8", alpha=0.85, label="TTFA (ms)")
ax.axhline(150.0, color="green", linestyle="--", label="Target TTFA (150ms)")
ax.set_xticks(x); ax.set_xticklabels(LEADERBOARD["Model"], rotation=25, ha="right", fontsize=8)
ax.set_title("CPU 2-Thread First-Audio Latency (ms)"); ax.grid(True, axis="y", alpha=0.3); ax.legend(fontsize=8)

plt.tight_layout()
chart_local = LOCAL_WORK / "reports" / "comparison_charts.png"
plt.savefig(chart_local, dpi=150)
sync_file_to_drive(chart_local, DRIVE_ROOT / "reports" / f"{CFG['dataset_source']}_{CFG['profile']}" / "comparison_charts.png")
plt.show()

# 2) 生成随机打乱的 A/B 盲听打分表（保存到 Google Drive 方便团队人工复核 CMOS）
blind_rows = []
rng_b = random.Random(CFG["seed"] + 99)
model_names = list(TRAINED_MODELS.keys())
preview_utts = [u for u in EVAL_SUITE if not u["is_val"]][:5] or EVAL_SUITE[:5]

print("\n================ 交互式试听面板（同一测试句对比各模型） ================")
for u in preview_utts:
    display(HTML(f"<h4>[{u['utt_id']}] {u['text']}</h4>"))
    shuffled = list(model_names)
    rng_b.shuffle(shuffled)
    for label_idx, m_name in enumerate(shuffled):
        rec = next(r for r in GENERATED_RECORDS[m_name] if r["utt_id"] == u["utt_id"])
        wav_p = rec["str_wav"] if rec["str_wav"] else rec["off_wav"]
        tag = f"Candidate_{chr(65 + label_idx)}"
        blind_rows.append({"utt_id": u["utt_id"], "text": u["text"], "blind_tag": tag, "true_model": m_name, "wav_path_on_drive": wav_p, "CMOS_score": "", "defect_tags": ""})
        print(f"  {tag} (流式/离线试听) — [展开盲听对照表查看真实模型名]:")
        display(Audio(filename=wav_p, rate=CFG["sample_rate"]))

blind_df = pd.DataFrame(blind_rows)
blind_local = LOCAL_WORK / "reports" / "blind_ab_listening_sheet.csv"
blind_df.to_csv(blind_local, index=False)
sync_file_to_drive(blind_local, DRIVE_ROOT / "reports" / f"{CFG['dataset_source']}_{CFG['profile']}" / "blind_ab_listening_sheet.csv")
print(f"[Saved] 盲听打分表与真实模型映射已同步到 Google 云盘。")


## 13. （可选扩展）Apache 2.0 教师离线造数脚本模板（构建 20h 单音色中英混读黄金集）

当你在 Phase 1 选出最佳流式主干后，如需打造**商用合规、单音色同时能说标准中文 + 英文 + 中英混读**的黄金训练集，可启用下方单元格（需先设置 `RUN_TEACHER_SYNTH = True` 并准备 1 段 5–15 秒合规参考干声放到云盘）。


In [ ]:
RUN_TEACHER_SYNTH = False  # 准备进入 Phase 2 造数时改为 True

if RUN_TEACHER_SYNTH:
    # 示例：利用 Apache 2.0 许可的 Qwen3-TTS 生成单音色双语语料并自动四重清洗写入 custom_bilingual
    # 参考文档: docs/03_teacher_distillation_pipeline.md
    OUT_CUSTOM_DIR = DRIVE_ROOT / "datasets" / "custom_bilingual"
    (OUT_CUSTOM_DIR / "wavs").mkdir(parents=True, exist_ok=True)
    print(f"教师造数输出目录已就绪: {OUT_CUSTOM_DIR}")
    print("步骤：1) 放入合规参考干声 ref.wav -> 2) 用 Qwen3-TTS / CosyVoice 批量合成 20-25h -> 3) 跑 Faster-Whisper CER=0 & Resemblyzer SIM>0.78 & UTMOS 过滤 -> 4) 切回 Cell 1 设置 DATASET_SOURCE='custom_bilingual' 开训！")
else:
    print("当前跳过教师离线造数（Phase 1 优先完成多主干对比实验）。")
